# Mondrian conformal HDF5 demo

This notebook runs Mondrian conformal analysis using saved CNN predictions, labels, and embeddings.

It supports two modes:

- real calibration/test mode, using `pred_cal`, `y_cal`, `emb_cal`, `pred_test`, `y_test`, and `emb_test`
- debug mode, where validation predictions are split into pseudo-calibration and pseudo-test subsets

Debug mode is useful for testing the pipeline, but must not be reported as final conformal performance.

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import iqr

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# Locate project root robustly from the current notebook location
cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd / "cbc_pe" / "src").exists():
    PROJECT_ROOT = cwd / "cbc_pe"
else:
    raise RuntimeError(
        f"Could not locate project root from cwd={cwd}. "
        "Expected to find a 'src/' directory."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


DATA_ROOT = Path("/data/vserrano/cbc_pe_data")
DATA_RESULTS = DATA_ROOT / "results"

dataset_id = "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000"
RESULTS_DIR = DATA_RESULTS / dataset_id

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("RESULTS_DIR exists:", RESULTS_DIR.exists())

## Run Selection

For the 100k architecture-search setup, calibration and test sets are not available. Use debug mode only for pipeline development.

For final reporting, use a prediction file produced from a 70/10/10/10 or similar train/validation/calibration/test split.

In [ ]:
# ---------------------------------------------------------------------
# Run selection
# ---------------------------------------------------------------------

RUN_ID = "500k_M00_baseline_emb64_seed123"
# RUN_ID = "100k_M00_baseline_emb64_seed123"
# RUN_ID = "100k_M00_baseline_emb64_seed124"
# RUN_ID = "100k_M04_pooldeep_emb128_pool4"

runs = {
    "500k_M00_baseline_emb64_seed123": {
        "dataset_id": "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000",
        "prediction_filename": (
            "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000"
            "_SimpleCNN_Baseline_simple_emb64_mse_MSELoss_seed123"
            "_train_val_cal_test_predictions_embeddings.npz"
        ),
    },
    "100k_M00_baseline_emb64_seed123": {
        "dataset_id": "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000",
        "prediction_filename": (
            "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
            "_SimpleCNN_Baseline_M00_simple_emb64_mse_MSELoss_seed123"
            "_train_val_predictions_embeddings.npz"
        ),
    },
    "100k_M00_baseline_emb64_seed124": {
        "dataset_id": "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000",
        "prediction_filename": (
            "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
            "_SimpleCNN_Baseline_M00_simple_emb64_mse_MSELoss_seed124"
            "_train_val_predictions_embeddings.npz"
        ),
    },
    "100k_M04_pooldeep_emb128_pool4": {
        "dataset_id": "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000",
        "prediction_filename": (
            "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
            "_SimpleCNN_PoolDeep_M04_emb128_pool4_deephead_MSELoss_seed123"
            "_train_val_predictions_embeddings.npz"
        ),
    },
}

run = runs[RUN_ID]

dataset_id = run["dataset_id"]
RESULTS_DIR = DATA_RESULTS / dataset_id
prediction_file = RESULTS_DIR / run["prediction_filename"]

assert prediction_file.exists(), prediction_file

print("RUN_ID:", RUN_ID)
print("dataset_id:", dataset_id)
print("RESULTS_DIR:", RESULTS_DIR)
print("prediction_file:", prediction_file)

data = np.load(prediction_file, allow_pickle=True)
print("NPZ keys:", data.files)

## Build calibration and test arrays

In [ ]:
def load_conformal_arrays(
    data,
    debug_split_val=False,
    debug_cal_fraction=0.5,
    debug_seed=123,
):
    """
    Load arrays needed for Mondrian/conformal.

    Preferred real mode:
        pred_cal, y_cal, emb_cal
        pred_test, y_test, emb_test

    Debug mode:
        If real cal/test are not available, split val into pseudo_cal/pseudo_test.
        This is only for debugging the notebook, not for final reporting.
    """

    files = set(data.files)

    has_real_cal_test = {
        "pred_cal",
        "y_cal",
        "emb_cal",
        "pred_test",
        "y_test",
        "emb_test",
    }.issubset(files)

    if has_real_cal_test:
        print("Using real cal/test arrays.")

        pred_cal = data["pred_cal"]
        y_cal = data["y_cal"]
        emb_cal = data["emb_cal"]

        pred_test = data["pred_test"]
        y_test = data["y_test"]
        emb_test = data["emb_test"]

        idx_cal = data["idx_cal"] if "idx_cal" in files else None
        idx_test = data["idx_test"] if "idx_test" in files else None

        mode = "real_cal_test"

    else:
        if not debug_split_val:
            raise KeyError(
                "No real cal/test arrays found in prediction file. "
                "Expected pred_cal/y_cal/emb_cal and pred_test/y_test/emb_test. "
                "For notebook debugging only, set debug_split_val=True."
            )

        required_val = {"pred_val", "y_val", "emb_val"}

        if not required_val.issubset(files):
            raise KeyError(
                "Cannot create debug split because pred_val/y_val/emb_val are missing."
            )

        print("WARNING: using validation split as pseudo cal/test.")
        print("This is only for debugging. Do not report these results as final.")

        pred_val = data["pred_val"]
        y_val = data["y_val"]
        emb_val = data["emb_val"]

        n_val = pred_val.shape[0]
        rng = np.random.default_rng(debug_seed)
        perm = rng.permutation(n_val)

        n_cal = int(debug_cal_fraction * n_val)

        cal_local = perm[:n_cal]
        test_local = perm[n_cal:]

        pred_cal = pred_val[cal_local]
        y_cal = y_val[cal_local]
        emb_cal = emb_val[cal_local]

        pred_test = pred_val[test_local]
        y_test = y_val[test_local]
        emb_test = emb_val[test_local]

        if "idx_val" in files:
            idx_val = data["idx_val"]
            idx_cal = idx_val[cal_local]
            idx_test = idx_val[test_local]
        else:
            idx_cal = cal_local
            idx_test = test_local

        mode = "debug_val_split"

    y_mean = data["y_mean"]
    y_std = data["y_std"]

    if "label_names" in files:
        label_names = data["label_names"].tolist()
    else:
        label_names = ["chirp_mass", "total_mass", "chi_eff"]

    return {
        "mode": mode,
        "pred_cal": pred_cal,
        "y_cal": y_cal,
        "emb_cal": emb_cal,
        "pred_test": pred_test,
        "y_test": y_test,
        "emb_test": emb_test,
        "idx_cal": idx_cal,
        "idx_test": idx_test,
        "y_mean": y_mean,
        "y_std": y_std,
        "label_names": label_names,
    }

In [ ]:
DEBUG_SPLIT_VAL = False  # Debug only. Do not report as final conformal performance.

arrays = load_conformal_arrays(
    data,
    debug_split_val=DEBUG_SPLIT_VAL,
    debug_cal_fraction=0.5,
    debug_seed=123,
)

mode = arrays["mode"]

pred_cal = arrays["pred_cal"]
y_cal = arrays["y_cal"]
emb_cal = arrays["emb_cal"]

pred_test = arrays["pred_test"]
y_test = arrays["y_test"]
emb_test = arrays["emb_test"]

idx_cal = arrays["idx_cal"]
idx_test = arrays["idx_test"]

y_mean = arrays["y_mean"]
y_std = arrays["y_std"]
label_names = arrays["label_names"]

print("mode:", mode)
print("pred_cal:", pred_cal.shape)
print("y_cal:", y_cal.shape)
print("emb_cal:", emb_cal.shape)
print("pred_test:", pred_test.shape)
print("y_test:", y_test.shape)
print("emb_test:", emb_test.shape)
print("y_mean:", y_mean)
print("y_std:", y_std)
print("label_names:", label_names)

if mode != "real_cal_test":
    print("WARNING: running in debug pseudo-cal/test mode. Do not report as final conformal performance.")

In [ ]:
# Sanity checks

assert pred_cal.ndim == 2
assert pred_test.ndim == 2
assert y_cal.ndim == 2
assert y_test.ndim == 2

assert pred_cal.shape == y_cal.shape
assert pred_test.shape == y_test.shape

assert pred_cal.shape[1] == len(label_names)
assert pred_test.shape[1] == len(label_names)

assert emb_cal.ndim == 2
assert emb_test.ndim == 2

assert emb_cal.shape[0] == pred_cal.shape[0]
assert emb_test.shape[0] == pred_test.shape[0]

assert y_mean.shape[0] == len(label_names)
assert y_std.shape[0] == len(label_names)
assert np.all(y_std > 0)

for name, arr in {
    "pred_cal": pred_cal,
    "y_cal": y_cal,
    "pred_test": pred_test,
    "y_test": y_test,
    "emb_cal": emb_cal,
    "emb_test": emb_test,
    "y_mean": y_mean,
    "y_std": y_std,
}.items():
    assert np.all(np.isfinite(arr)), f"{name} contains NaN or inf"

if mode != "real_cal_test":
    print("WARNING: notebook is running in debug mode:", mode)
    print("Do not use these results as final conformal/Mondrian results.")

print("All sanity checks passed.")

In [ ]:
def inverse_standardize(y_std_values, y_mean, y_std):
    return y_std_values * y_std + y_mean


y_cal_phys = inverse_standardize(y_cal, y_mean, y_std)
y_test_phys = inverse_standardize(y_test, y_mean, y_std)

pred_cal_phys = inverse_standardize(pred_cal, y_mean, y_std)
pred_test_phys = inverse_standardize(pred_test, y_mean, y_std)

label_ranges_phys = {
    label: np.max(y_test_phys[:, j]) - np.min(y_test_phys[:, j])
    for j, label in enumerate(label_names)
}

label_ranges_phys

## Start Mondrian: Build the DF

In [ ]:
from src.conformal.pipeline import run_mondrian_regression

confidence_level = 0.90

if mode == "real_cal_test":
    n_bins_grid = [4, 6, 8, 12, 16, 24, 32, 48]
else:
    n_bins_grid = [4, 6, 8, 12, 16, 24]

taxonomy_modes = ["prediction", "difficulty"]
interval_modes = ["symmetric", "asymmetric"]

n_neighbors = 5
min_samples_per_bin = 20 if mode == "real_cal_test" else 10

rows = []
all_results = {}



for taxonomy_mode in taxonomy_modes:
    for interval_mode in interval_modes:
        for n_bins in n_bins_grid:

            kwargs = dict(
                pred_cal=pred_cal,
                pred_test=pred_test,
                y_cal=y_cal,
                y_test=y_test,
                n_bins=n_bins,
                confidence_level=confidence_level,
                apply_jitter=True,
                interval_mode=interval_mode,
                taxonomy_mode=taxonomy_mode,
                min_samples_per_bin=min_samples_per_bin,
                tolerance_sigmas=(1, 2, 3),
            )

            if taxonomy_mode == "difficulty":
                kwargs.update(
                    cal_embedding=emb_cal,
                    target_embedding=emb_test,
                    n_neighbors=n_neighbors,
                )

            result = run_mondrian_regression(**kwargs)
            all_results[(taxonomy_mode, interval_mode, n_bins)] = result

            metrics = result.metrics

            widths_std = result.upper - result.lower
            widths_phys = widths_std * y_std

            for j, label in enumerate(label_names):
                # ------------------------------------------------------------
                # Existing p-value based local undercoverage diagnostic
                # ------------------------------------------------------------
                n_bad_bins_p005 = int(
                    np.nansum(metrics["bin_undercoverage_pvalue"][:, j] < 0.05)
                )

                # ------------------------------------------------------------
                # New: local 2-sigma bin-wise validity diagnostics
                # ------------------------------------------------------------
                coverage_bin = metrics["coverage_per_bin"][:, j]
                count_bin = metrics["counts_per_bin"][:, j]

                bin_tol = metrics["bin_tolerance_normal"]

                bin_2sigma_low = bin_tol["2sigma_low"][:, j]
                bin_2sigma_high = bin_tol["2sigma_high"][:, j]
                bin_2sigma_width = bin_tol["2sigma_width"][:, j]

                valid_bins = count_bin > 0

                bin_within_2sigma = (
                    (coverage_bin >= bin_2sigma_low)
                    & (coverage_bin <= bin_2sigma_high)
                    & valid_bins
                )

                bin_under_2sigma = (
                    (coverage_bin < bin_2sigma_low)
                    & valid_bins
                )

                bin_over_2sigma = (
                    (coverage_bin > bin_2sigma_high)
                    & valid_bins
                )

                all_bins_within_2sigma = bool(np.all(bin_within_2sigma[valid_bins]))

                n_bins_outside_2sigma = int(np.sum(~bin_within_2sigma[valid_bins]))
                outside_bin_fraction_2sigma = float(np.mean(~bin_within_2sigma[valid_bins]))

                n_bins_under_2sigma = int(np.sum(bin_under_2sigma[valid_bins]))
                under_bin_fraction_2sigma = float(np.mean(bin_under_2sigma[valid_bins]))

                n_bins_over_2sigma = int(np.sum(bin_over_2sigma[valid_bins]))
                over_bin_fraction_2sigma = float(np.mean(bin_over_2sigma[valid_bins]))

                min_bin_2sigma_low = float(np.nanmin(bin_2sigma_low[valid_bins]))
                max_bin_2sigma_high = float(np.nanmax(bin_2sigma_high[valid_bins]))
                median_bin_2sigma_width = float(np.nanmedian(bin_2sigma_width[valid_bins]))

                row = {
                    "mode": mode,
                    "taxonomy_mode": taxonomy_mode,
                    "interval_mode": interval_mode,
                    "n_bins": n_bins,
                    "label": label,
                    "label_index": j,

                    "n_samples": int(metrics["n_samples_per_label"][j]),
                    "covered_count": int(metrics["covered_count_global"][j]),

                    "global_coverage": metrics["global_coverage"][j],
                    "miscoverage": metrics["miscoverage"][j],
                    "global_coverage_gap": metrics["global_coverage_gap"][j],
                    "global_undercoverage_pvalue": metrics["global_undercoverage_pvalue"][j],

                    "global_mean_width_std": metrics["global_mean_width"][j],
                    "global_median_width_std": metrics["global_median_width"][j],

                    "global_mean_width_phys": np.mean(widths_phys[:, j]),
                    "global_median_width_phys": np.median(widths_phys[:, j]),
                    "global_iqr_width_phys": iqr(widths_phys[:, j]),

                    "normalized_median_width_range": (
                        np.median(widths_phys[:, j]) / label_ranges_phys[label]
                    ),

                    "min_coverage_per_bin": metrics["min_coverage_per_label"][j],
                    "max_undercoverage_gap": metrics["max_undercoverage_gap"][j],

                    "min_count_per_bin": int(np.nanmin(metrics["counts_per_bin"][:, j])),
                    "max_count_per_bin": int(np.nanmax(metrics["counts_per_bin"][:, j])),

                    "n_bad_bins_p005": n_bad_bins_p005,
                    "bad_bin_fraction": n_bad_bins_p005 / n_bins,

                    # Local 2-sigma bin-wise validity
                    "all_bins_within_2sigma": all_bins_within_2sigma,
                    "n_bins_outside_2sigma": n_bins_outside_2sigma,
                    "outside_bin_fraction_2sigma": outside_bin_fraction_2sigma,

                    "n_bins_under_2sigma": n_bins_under_2sigma,
                    "under_bin_fraction_2sigma": under_bin_fraction_2sigma,

                    "n_bins_over_2sigma": n_bins_over_2sigma,
                    "over_bin_fraction_2sigma": over_bin_fraction_2sigma,

                    "min_bin_2sigma_low": min_bin_2sigma_low,
                    "max_bin_2sigma_high": max_bin_2sigma_high,
                    "median_bin_2sigma_width": median_bin_2sigma_width,

                    "global_lower_miss_rate": metrics["global_lower_miss_rate"][j],
                    "global_upper_miss_rate": metrics["global_upper_miss_rate"][j],
                    "global_tail_miss_imbalance": metrics["global_tail_miss_imbalance"][j],
                }

                global_tol = metrics["global_tolerance_normal"]

                for k in [1, 2, 3]:
                    low = global_tol[f"{k}sigma_low"][j]
                    high = global_tol[f"{k}sigma_high"][j]
                    width = global_tol[f"{k}sigma_width"][j]

                    row[f"global_tol_{k}sigma_low"] = low
                    row[f"global_tol_{k}sigma_high"] = high
                    row[f"global_tol_{k}sigma_width"] = width
                    row[f"global_within_{k}sigma"] = bool(low <= metrics["global_coverage"][j] <= high)

                rows.append(row)

summary_df = pd.DataFrame(rows)


print("summary_df shape:", summary_df.shape)
summary_df.head()



In [ ]:
column_descriptions = {
    # Identity / configuration
    "mode": "Evaluation mode, e.g. real_cal_test.",
    "taxonomy_mode": "Mondrian taxonomy used to define bins: prediction or difficulty.",
    "interval_mode": "Conformal interval type: symmetric or asymmetric.",
    "n_bins": "Number of Mondrian bins.",
    "label": "Target label: chirp_mass, total_mass, chi_eff.",
    "label_index": "Index of the target label.",

    # Global coverage
    "n_samples": "Number of test samples evaluated.",
    "covered_count": "Number of test samples whose true value is inside the interval.",
    "global_coverage": "Empirical coverage on the test set.",
    "miscoverage": "1 - global_coverage.",
    "global_coverage_gap": "Absolute gap between empirical coverage and target coverage.",
    "global_undercoverage_pvalue": "Binomial p-value for global undercoverage.",

    # Widths
    "global_mean_width_std": "Mean interval width in standardized target units.",
    "global_median_width_std": "Median interval width in standardized target units.",
    "global_mean_width_phys": "Mean interval width in physical units.",
    "global_median_width_phys": "Median interval width in physical units.",
    "global_iqr_width_phys": "IQR of interval widths in physical units.",
    "normalized_median_width_range": "Median physical width normalized by the label physical range.",

    # Local bin summary
    "min_coverage_per_bin": "Minimum empirical coverage among bins.",
    "max_undercoverage_gap": "Largest undercoverage gap among bins relative to target coverage.",
    "min_count_per_bin": "Minimum number of test samples in any bin.",
    "max_count_per_bin": "Maximum number of test samples in any bin.",
    "n_bad_bins_p005": "Number of bins flagged as significantly undercovered at p < 0.05.",
    "bad_bin_fraction": "Fraction of bins flagged as significantly undercovered.",

    # Tail balance
    "global_lower_miss_rate": "Fraction of samples missed below the lower bound.",
    "global_upper_miss_rate": "Fraction of samples missed above the upper bound.",
    "global_tail_miss_imbalance": "Absolute imbalance between lower and upper miss rates.",

    # Global binomial tolerance
    "global_tol_1sigma_low": "Lower 1σ tolerance for global coverage.",
    "global_tol_1sigma_high": "Upper 1σ tolerance for global coverage.",
    "global_tol_1sigma_width": "Half-width of 1σ global tolerance.",
    "global_within_1sigma": "Whether global coverage is inside 1σ tolerance.",
    "global_tol_2sigma_low": "Lower 2σ tolerance for global coverage.",
    "global_tol_2sigma_high": "Upper 2σ tolerance for global coverage.",
    "global_tol_2sigma_width": "Half-width of 2σ global tolerance.",
    "global_within_2sigma": "Whether global coverage is inside 2σ tolerance.",
    "global_tol_3sigma_low": "Lower 3σ tolerance for global coverage.",
    "global_tol_3sigma_high": "Upper 3σ tolerance for global coverage.",
    "global_tol_3sigma_width": "Half-width of 3σ global tolerance.",
    "global_within_3sigma": "Whether global coverage is inside 3σ tolerance.",
}

In [ ]:
new_local_cols = [
    "all_bins_within_2sigma",
    "n_bins_outside_2sigma",
    "outside_bin_fraction_2sigma",
    "n_bins_under_2sigma",
    "under_bin_fraction_2sigma",
    "n_bins_over_2sigma",
    "over_bin_fraction_2sigma",
    "min_bin_2sigma_low",
    "max_bin_2sigma_high",
    "median_bin_2sigma_width",
]

summary_df[
    [
        "label",
        "taxonomy_mode",
        "interval_mode",
        "n_bins",
        "global_coverage",
        "global_within_2sigma",
        "min_coverage_per_bin",
        "max_undercoverage_gap",
    ]
    + new_local_cols
].head(20)

## Save the results

In [ ]:
mondrian_results_dir = RESULTS_DIR / "mondrian"
mondrian_results_dir.mkdir(parents=True, exist_ok=True)

prediction_stem = prediction_file.stem

summary_path = mondrian_results_dir / f"{prediction_stem}_mondrian_summary_{mode}.csv"

summary_df.to_csv(summary_path, index=False)

print("Saved:", summary_path)

## Ranking Configurations

### Filtering the configurations

In [ ]:
# ============================================================
# Final Mondrian configuration selection
# ============================================================

MIN_COUNT_PER_BIN = 200
WIDTH_TIE_FRACTION = 0.02

MAX_UNDER_BIN_FRACTION_EFFICIENT = 0.10
MAX_UNDERCOVERAGE_GAP_EFFICIENT = 0.05

TOP_K = 5

required_cols = [
    "label",
    "label_index",
    "taxonomy_mode",
    "interval_mode",
    "n_bins",
    "global_coverage",
    "global_within_2sigma",
    "min_count_per_bin",
    "min_coverage_per_bin",
    "n_bins_under_2sigma",
    "under_bin_fraction_2sigma",
    "max_undercoverage_gap",
    "global_median_width_phys",
    "global_lower_miss_rate",
    "global_upper_miss_rate",
    "global_tail_miss_imbalance",
    "n_bad_bins_p005",
    "bad_bin_fraction",
]

missing_cols = [
    col for col in required_cols
    if col not in summary_df.columns
]

if missing_cols:
    raise KeyError(
        f"Missing required columns in summary_df: {missing_cols}"
    )


def get_base_candidates(summary_df, label):
    """
    Return configurations satisfying the common hard requirements.

    These requirements are shared by the conservative and efficient
    selection policies.
    """
    candidates = summary_df[
        (summary_df["label"] == label)
        & (summary_df["global_within_2sigma"])
        & (summary_df["min_count_per_bin"] >= MIN_COUNT_PER_BIN)
    ].copy()

    return candidates

In [ ]:
def select_conservative(summary_df, label):
    candidates = get_base_candidates(summary_df, label)

    if candidates.empty:
        raise ValueError(f"No globally valid candidates for label={label}")

    zero_under = candidates[
        candidates["n_bins_under_2sigma"] == 0
    ].copy()

    if not zero_under.empty:
        pool = zero_under
        policy = "conservative_zero_under_bins_2sigma"
    else:
        # Fallback only if no configuration has zero undercovered bins.
        min_under = candidates["n_bins_under_2sigma"].min()

        pool = candidates[
            candidates["n_bins_under_2sigma"] == min_under
        ].copy()

        policy = "conservative_best_available"

    # First restrict to configurations near the minimum physical width.
    min_width = pool["global_median_width_phys"].min()
    width_limit = min_width * (1.0 + WIDTH_TIE_FRACTION)

    tied = pool[
        pool["global_median_width_phys"] <= width_limit
    ].copy()

    tied["min_width_for_label"] = min_width
    tied["width_limit"] = width_limit
    tied["relative_width_excess"] = (
        tied["global_median_width_phys"] / min_width - 1.0
    )

    # Within the width tie, prefer more bins.
    ranked = tied.sort_values(
        by=[
            "n_bins",
            "max_undercoverage_gap",
            "global_tail_miss_imbalance",
            "global_median_width_phys",
        ],
        ascending=[
            False,
            True,
            True,
            True,
        ],
    )

    selected = ranked.iloc[0].copy()
    selected["selection_policy"] = policy

    return selected, ranked


def select_efficient(summary_df, label):
    candidates = get_base_candidates(summary_df, label)

    candidates = candidates[
        (candidates["under_bin_fraction_2sigma"]
         <= MAX_UNDER_BIN_FRACTION_EFFICIENT)
        & (candidates["max_undercoverage_gap"]
           <= MAX_UNDERCOVERAGE_GAP_EFFICIENT)
    ].copy()

    if candidates.empty:
        raise ValueError(f"No efficient valid candidates for label={label}")

    min_width = candidates["global_median_width_phys"].min()
    width_limit = min_width * (1.0 + WIDTH_TIE_FRACTION)

    tied = candidates[
        candidates["global_median_width_phys"] <= width_limit
    ].copy()

    tied["min_width_for_label"] = min_width
    tied["width_limit"] = width_limit
    tied["relative_width_excess"] = (
        tied["global_median_width_phys"] / min_width - 1.0
    )

    ranked = tied.sort_values(
        by=[
            "n_bins",
            "n_bins_under_2sigma",
            "max_undercoverage_gap",
            "global_tail_miss_imbalance",
            "global_median_width_phys",
        ],
        ascending=[
            False,
            True,
            True,
            True,
            True,
        ],
    )

    selected = ranked.iloc[0].copy()
    selected["selection_policy"] = "efficient_local_tolerance"

    return selected, ranked

In [ ]:
conservative_rows = []
efficient_rows = []

conservative_top_tables = []
efficient_top_tables = []

for label in label_names:
    conservative_selected, conservative_ranked = select_conservative(
        summary_df,
        label,
    )

    efficient_selected, efficient_ranked = select_efficient(
        summary_df,
        label,
    )

    conservative_rows.append(conservative_selected)
    efficient_rows.append(efficient_selected)

    # Add policy information to every row in the ranked candidate tables.
    conservative_policy = conservative_selected["selection_policy"]
    efficient_policy = efficient_selected["selection_policy"]

    conservative_top_tables.append(
        conservative_ranked
        .head(TOP_K)
        .assign(
            final_policy="conservative",
            selection_policy=conservative_policy,
        )
    )

    efficient_top_tables.append(
        efficient_ranked
        .head(TOP_K)
        .assign(
            final_policy="efficient",
            selection_policy=efficient_policy,
        )
    )

conservative_by_label = pd.DataFrame(conservative_rows)
efficient_by_label = pd.DataFrame(efficient_rows)

top_by_label = pd.concat(
    conservative_top_tables + efficient_top_tables,
    ignore_index=True,
)

# The conservative selection is the primary configuration used
# by the rest of the notebook.
final_by_label = conservative_by_label.copy()

assert "label_index" in final_by_label.columns

In [ ]:
selection_display_cols = [
    "label",
    "selection_policy",
    "taxonomy_mode",
    "interval_mode",
    "n_bins",
    "global_coverage",
    "global_within_2sigma",
    "min_coverage_per_bin",
    "n_bins_under_2sigma",
    "under_bin_fraction_2sigma",
    "n_bad_bins_p005",
    "bad_bin_fraction",
    "max_undercoverage_gap",
    "global_median_width_phys",
    "relative_width_excess",
    "global_lower_miss_rate",
    "global_upper_miss_rate",
    "global_tail_miss_imbalance",
    "min_count_per_bin",
]

selection_comparison_df = pd.concat(
    [
        conservative_by_label.assign(final_policy="conservative"),
        efficient_by_label.assign(final_policy="efficient"),
    ],
    ignore_index=True,
)

selection_comparison_df[
    ["final_policy"] + selection_display_cols
].sort_values(["label", "final_policy"])

In [ ]:
top_display_cols = [
    "final_policy",
    "label",
    "selection_policy",
    "taxonomy_mode",
    "interval_mode",
    "n_bins",
    "global_coverage",
    "global_within_2sigma",
    "min_coverage_per_bin",
    "n_bins_under_2sigma",
    "under_bin_fraction_2sigma",
    "n_bad_bins_p005",
    "bad_bin_fraction",
    "max_undercoverage_gap",
    "global_median_width_phys",
    "relative_width_excess",
    "global_tail_miss_imbalance",
    "min_count_per_bin",
]

top_by_label[
    top_display_cols
].sort_values(
    ["label", "final_policy", "global_median_width_phys"]
)

In [ ]:
# Primary final configuration: conservative selection

final_display_cols = [
    "label",
    "selection_policy",
    "taxonomy_mode",
    "interval_mode",
    "n_bins",
    "global_coverage",
    "global_within_2sigma",
    "min_coverage_per_bin",
    "n_bins_under_2sigma",
    "under_bin_fraction_2sigma",
    "n_bad_bins_p005",
    "bad_bin_fraction",
    "max_undercoverage_gap",
    "global_median_width_phys",
    "relative_width_excess",
    "global_lower_miss_rate",
    "global_upper_miss_rate",
    "global_tail_miss_imbalance",
    "min_count_per_bin",
]

final_by_label[
    final_display_cols
].sort_values("label")

## Criterio de selección de la configuración Mondrian final

La selección final se trata como un problema multiobjetivo. No se busca
únicamente minimizar la anchura de los intervalos ni maximizar el número
de bins de manera aislada. Un mayor número de bins aporta mayor
adaptabilidad local, pero también reduce el número de muestras disponible
en cada bin y puede aumentar la variabilidad de la cobertura estimada.

Todas las configuraciones candidatas deben cumplir primero:

- cobertura global compatible con el nivel nominal dentro de 2σ;
- al menos 200 muestras en el bin menos poblado;
- evaluación mediante conjuntos independientes de calibración y test.

Se consideran dos políticas de selección.

### Política conservadora

La política conservadora es la selección principal del análisis.

1. Se priorizan las configuraciones con cero bins infracubiertos respecto
   a la banda local de 2σ.
2. Si no existe ninguna configuración con cero bins infracubiertos, se
   conserva el menor número de bins problemáticos disponible.
3. Dentro de ese conjunto, se identifica la menor anchura física mediana.
4. Se consideran equivalentes en anchura las configuraciones situadas
   dentro de un 2% de la anchura mínima.
5. Sólo dentro de ese empate de anchura se favorece el mayor número de bins.

Por tanto, el número de bins actúa como criterio de desempate después de
garantizar la validez global, la estabilidad estadística y la cobertura
local.

### Política eficiente

La política eficiente se conserva como análisis de sensibilidad. Permite:

- una fracción máxima del 10% de bins infracubiertos a 2σ;
- un gap máximo de infracobertura de 0.05.

Entre las configuraciones que cumplen estas restricciones se aplica la
misma tolerancia del 2% en anchura y se favorece el mayor número de bins.
Esta política permite estudiar el compromiso entre intervalos más
especializados y una validez local menos estricta.

### Papel del test binomial

El número de bins con p-value de infracobertura inferior a 0.05 se
mantiene como diagnóstico complementario. La banda local de 2σ es el
criterio principal de selección. Ambos indicadores estudian la cobertura
local, pero no tienen por qué clasificar exactamente los mismos bins como
problemáticos.

Los resultados numéricos no se escriben manualmente en esta sección.
Se generan directamente en las tablas siguientes para evitar que el texto
quede desactualizado al repetir el análisis.

In [ ]:
final_report_cols = [
    "label",
    "taxonomy_mode",
    "interval_mode",
    "n_bins",
    "global_coverage",
    "global_within_2sigma",
    "min_coverage_per_bin",
    "n_bins_under_2sigma",
    "under_bin_fraction_2sigma",
    "n_bad_bins_p005",
    "bad_bin_fraction",
    "max_undercoverage_gap",
    "global_median_width_phys",
    "relative_width_excess",
    "global_lower_miss_rate",
    "global_upper_miss_rate",
    "global_tail_miss_imbalance",
    "min_count_per_bin",
    "selection_policy",
]

final_report_df = (
    final_by_label[final_report_cols]
    .sort_values("label")
    .reset_index(drop=True)
)

final_report_df

In [ ]:
sensitivity_report_df = selection_comparison_df[
    ["final_policy"] + final_report_cols
].copy()

sensitivity_report_df

**Nota:** la política principal prioriza la ausencia de infracobertura
local sobre la maximización del número de bins. Por tanto, una
configuración con menos bins puede ser preferible si ofrece intervalos
más estables y localmente válidos. La política eficiente se conserva
como análisis de sensibilidad para cuantificar el coste de exigir esta
validez local más estricta.

## Plots

In [ ]:
plot_labels = {
    "chirp_mass": r"$\mathcal{M}$",
    "total_mass": r"$M_{\mathrm{tot}}$",
    "chi_eff": r"$\chi_{\mathrm{eff}}$",
}

plot_colors = {
    "symmetric": "blueviolet",
    "asymmetric": "indigo",
}


def plot_global_coverage_vs_bins(
    summary_df,
    label,
    taxonomy_modes,
    interval_modes,
    confidence_level,
    plot_labels,
    plot_colors=None,
):
    df_label = summary_df[summary_df["label"] == label]

    fig, axes = plt.subplots(
        1,
        len(taxonomy_modes),
        figsize=(14, 5),
        sharey=True,
        constrained_layout=True,
    )

    axes = np.atleast_1d(axes)

    for ax, taxonomy_mode in zip(axes, taxonomy_modes):
        df_tax = df_label[df_label["taxonomy_mode"] == taxonomy_mode]

        if df_tax.empty:
            ax.set_title(f"{taxonomy_mode} (no data)")
            continue

        # Global tolerance is constant for this label.
        low_1 = df_tax["global_tol_1sigma_low"].iloc[0]
        high_1 = df_tax["global_tol_1sigma_high"].iloc[0]
        low_2 = df_tax["global_tol_2sigma_low"].iloc[0]
        high_2 = df_tax["global_tol_2sigma_high"].iloc[0]
        low_3 = df_tax["global_tol_3sigma_low"].iloc[0]
        high_3 = df_tax["global_tol_3sigma_high"].iloc[0]

        ax.axhspan(low_1, high_1, alpha=0.40, label=r"$1\sigma$")
        ax.axhspan(low_2, high_2, alpha=0.35, label=r"$2\sigma$")
        ax.axhspan(low_3, high_3, alpha=0.30, label=r"$3\sigma$")

        ax.axhline(
            confidence_level,
            color="black",
            linestyle="--",
            linewidth=1,
            label=rf"C.L. = {int(confidence_level * 100)}%",
        )

        for interval_mode in interval_modes:
            df_mode = (
                df_tax[df_tax["interval_mode"] == interval_mode]
                .sort_values("n_bins")
            )

            color = None if plot_colors is None else plot_colors.get(interval_mode)

            ax.plot(
                df_mode["n_bins"],
                df_mode["global_coverage"],
                marker="o",
                label=interval_mode,
                color=color,
            )

        ax.set_xticks(sorted(df_tax["n_bins"].unique()))
        ax.set_xlabel(r"$n_{\mathrm{bins}}$", fontsize=13)
        ax.set_title(taxonomy_mode)
        ax.grid(alpha=0.25)

    axes[0].set_ylabel("Global coverage", fontsize=13)

    fig.suptitle(f"Coverage vs bins ({plot_labels[label]})", fontsize=16)

    handles, labels_legend = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels_legend,
        loc="upper left",
        bbox_to_anchor=(0.82, 1.08),
        ncol=2,
    )

    plt.show()

In [ ]:
for label in label_names:
    plot_global_coverage_vs_bins(
        summary_df=summary_df,
        label=label,
        taxonomy_modes=taxonomy_modes,
        interval_modes=interval_modes,
        confidence_level=confidence_level,
        plot_labels=plot_labels,
        plot_colors=plot_colors,
    )

In [ ]:
def plot_width_vs_bins(
    summary_df,
    label,
    taxonomy_modes,
    interval_modes,
    plot_labels,
    width_column="global_median_width_std",
    plot_colors=None,
):
    df_label = summary_df[summary_df["label"] == label]

    fig, axes = plt.subplots(
        1,
        len(taxonomy_modes),
        figsize=(14, 5),
        sharey=True,
        constrained_layout=True,
    )

    axes = np.atleast_1d(axes)

    for ax, taxonomy_mode in zip(axes, taxonomy_modes):
        df_tax = df_label[df_label["taxonomy_mode"] == taxonomy_mode]

        for interval_mode in interval_modes:
            df_mode = (
                df_tax[df_tax["interval_mode"] == interval_mode]
                .sort_values("n_bins")
            )

            color = None if plot_colors is None else plot_colors.get(interval_mode)

            ax.plot(
                df_mode["n_bins"],
                df_mode[width_column],
                marker="o",
                label=interval_mode,
                color=color,
            )

        ax.set_title(taxonomy_mode)
        ax.set_xlabel(r"$n_{\mathrm{bins}}$", fontsize=13)
        ax.set_xticks(sorted(df_tax["n_bins"].unique()))
        ax.grid(alpha=0.25)

    axes[0].set_ylabel(width_column, fontsize=13)
    fig.suptitle(f"Interval width vs bins ({plot_labels[label]})", fontsize=16)

    handles, labels_legend = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels_legend,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.08),
        ncol=2,
    )

    plt.show()

In [ ]:
for label in label_names:
    plot_width_vs_bins(
        summary_df=summary_df,
        label=label,
        taxonomy_modes=taxonomy_modes,
        interval_modes=interval_modes,
        plot_labels=plot_labels,
        width_column="global_median_width_phys",
        plot_colors=plot_colors,
    )

In [ ]:
def plot_coverage_width_tradeoff(
    summary_df,
    label,
    taxonomy_modes,
    interval_modes,
    confidence_level,
    plot_labels,
    width_column="global_median_width_std",
):
    df_label = summary_df[summary_df["label"] == label]

    scatter_color = {

    }

    plt.figure(figsize=(16, 10))

    for taxonomy_mode in taxonomy_modes:
        for interval_mode in interval_modes:
            df_mode = df_label[
                (df_label["taxonomy_mode"] == taxonomy_mode) &
                (df_label["interval_mode"] == interval_mode)
            ]

            plt.scatter(
                df_mode[width_column],
                df_mode["global_coverage"],
                s=350,
                label=f"{taxonomy_mode}-{interval_mode}",
            )

            for _, row in df_mode.iterrows():
                plt.text(
                    row[width_column],
                    row["global_coverage"],
                    str(row["n_bins"]),
                    fontsize=14,
                    ha="center",
                    va="center",
                    color="white",
                )

    plt.axhline(confidence_level, color="black", linestyle="--", linewidth=1)
    plt.xlabel(width_column)
    plt.ylabel("Global coverage")
    plt.title(f"Coverage-efficiency tradeoff ({plot_labels[label]})", fontsize=18)
    plt.grid(alpha=0.25)
    plt.legend(fontsize=14)
    plt.show()

In [ ]:
for label in label_names:
    plot_coverage_width_tradeoff(
        summary_df=summary_df[
            (summary_df["global_within_2sigma"])
            & (summary_df["min_count_per_bin"] >= MIN_COUNT_PER_BIN)
        ].copy(),
        label=label,
        taxonomy_modes=taxonomy_modes,
        interval_modes=interval_modes,
        confidence_level=confidence_level,
        plot_labels=plot_labels,
        width_column="global_median_width_phys",
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_coverage_and_width_per_bin(
    all_results,
    key,
    label_idx,
    label_names,
    plot_labels,
    confidence_level,
    width_stat="mean",          # "mean" or "median"
    width_space="std",          # "std" or "phys"
    y_std=None,                 # required if width_space="phys"
    coverage_ylim=(0.85, 0.95),
    width_ylim=None,
    annotate_counts=True,
):
    result = all_results[key]
    metrics = result.metrics

    label = label_names[label_idx]
    title_label = plot_labels[label] if label in plot_labels else label

    coverage_per_bin = metrics["coverage_per_bin"][:, label_idx]
    counts_per_bin = metrics["counts_per_bin"][:, label_idx]
    bin_tol = metrics["bin_tolerance_normal"]

    n_bins = len(coverage_per_bin)
    x = np.arange(n_bins)

    # ------------------------------------------------------------------
    # Width per bin
    # ------------------------------------------------------------------
    width_key = f"{width_stat}_width_per_bin"

    if width_key in metrics:
        width_per_bin = metrics[width_key][:, label_idx].astype(float)
    else:
        # Fallback: compute from test intervals
        bin_indices_test = result.bin_indices_test
        widths = result.upper - result.lower  # (n_test, n_labels)

        width_per_bin = np.full(n_bins, np.nan)

        for b in range(n_bins):
            mask = bin_indices_test[:, label_idx] == b

            if np.sum(mask) == 0:
                continue

            if width_stat == "mean":
                width_per_bin[b] = np.mean(widths[mask, label_idx])
            elif width_stat == "median":
                width_per_bin[b] = np.median(widths[mask, label_idx])
            else:
                raise ValueError("width_stat must be 'mean' or 'median'.")

    if width_stat not in ["mean", "median"]:
        raise ValueError("width_stat must be 'mean' or 'median'.")

    if width_space == "phys":
        if y_std is None:
            raise ValueError("y_std must be provided when width_space='phys'.")

        width_per_bin = width_per_bin * y_std[label_idx]
        width_ylabel = f"{width_stat.capitalize()} interval width [{label}]"

    elif width_space == "std":
        width_ylabel = f"{width_stat.capitalize()} interval width [standardized]"

    else:
        raise ValueError("width_space must be 'std' or 'phys'.")

    # ------------------------------------------------------------------
    # Coverage bands
    # ------------------------------------------------------------------
    low_1 = bin_tol["1sigma_low"][:, label_idx]
    low_2 = bin_tol["2sigma_low"][:, label_idx]
    low_3 = bin_tol["3sigma_low"][:, label_idx]

    high_1 = bin_tol["1sigma_high"][:, label_idx]
    high_2 = bin_tol["2sigma_high"][:, label_idx]
    high_3 = bin_tol["3sigma_high"][:, label_idx]

    # ------------------------------------------------------------------
    # Plot
    # ------------------------------------------------------------------
    fig, ax1 = plt.subplots(figsize=(9.5, 5.2))

    band_color = "tab:red"

    ax1.fill_between(
        x, low_3, high_3,
        color=band_color,
        alpha=0.10,
        label=r"Nominal 3$\sigma$",
        zorder=1,
    )

    ax1.fill_between(
        x, low_2, high_2,
        color=band_color,
        alpha=0.15,
        label=r"Nominal 2$\sigma$",
        zorder=2,
    )

    ax1.fill_between(
        x, low_1, high_1,
        color=band_color,
        alpha=0.20,
        label=r"Nominal 1$\sigma$",
        zorder=3,
    )

    ax1.plot(
        x,
        coverage_per_bin,
        marker="o",
        color="maroon",
        label="Empirical coverage",
        zorder=4,
    )

    ax1.axhline(
        confidence_level,
        linestyle="--",
        alpha=0.6,
        linewidth=1.0,
        color="black",
        label=rf"C.L. = {confidence_level}",
    )

    if annotate_counts:
        for i, n in enumerate(counts_per_bin):
            if np.isfinite(coverage_per_bin[i]):
                ax1.text(
                    i,
                    coverage_per_bin[i] + 0.005,
                    str(int(n)),
                    ha="center",
                    fontsize=9,
                    color="black",
                )

    ax1.set_xlabel("Bin index")
    ax1.set_ylabel("Coverage")
    ax1.set_ylim(*coverage_ylim)
    ax1.grid(alpha=0.25)

    # ------------------------------------------------------------------
    # Right axis: interval width
    # ------------------------------------------------------------------
    ax2 = ax1.twinx()

    ax2.plot(
        x,
        width_per_bin,
        marker="s",
        linestyle="-",
        color="tab:blue",
        label=f"{width_stat} interval width",
        zorder=5,
    )

    ax2.set_ylabel(width_ylabel)

    if width_ylim is not None:
        ax2.set_ylim(*width_ylim)

    ax1.set_title(
        f"Coverage and interval width per bin | {title_label} | {key}"
    )

    # Combine legends
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()

    ax1.legend(
        handles1 + handles2,
        labels1 + labels2,
        ncols=2,
        loc="lower right",
    )

    plt.tight_layout()
    plt.show()

    return width_per_bin

## Final candidate configuration

In [ ]:
def inspect_configuration(
    all_results,
    key,
    label_idx,
    label_names,
):
    result = all_results[key]
    metrics = result.metrics
    label = label_names[label_idx]

    print("Configuration:", key)
    print("Label:", label)

    print("\nCounts per bin:")
    print(metrics["counts_per_bin"][:, label_idx])

    print("\nCoverage per bin:")
    print(np.round(metrics["coverage_per_bin"][:, label_idx], 3))

    print("\nCovered count per bin:")
    print(metrics["covered_count_per_bin"][:, label_idx])

    print("\nBin undercoverage p-values:")
    print(np.round(metrics["bin_undercoverage_pvalue"][:, label_idx], 4))

    print("\nLower miss rate per bin:")
    print(np.round(metrics["lower_miss_rate_per_bin"][:, label_idx], 3))

    print("\nUpper miss rate per bin:")
    print(np.round(metrics["upper_miss_rate_per_bin"][:, label_idx], 3))

    print("\nInterval offsets per bin:")
    print(np.round(result.intervals[label_idx], 4))

    print("\nCalibrator bin counts:")
    print(result.calibrator.bin_counts_[label_idx])

    if hasattr(result.calibrator, "quantile_indices_"):
        print("\nQuantile indices:")
        print(result.calibrator.quantile_indices_[label_idx])

In [ ]:
for row in final_by_label.itertuples():
    key = (row.taxonomy_mode, row.interval_mode, row.n_bins)
    label_idx = int(row.label_index)
    label = label_names[label_idx]

    print("=" * 80)
    print(f"Selected configuration for label: {label}")
    print(f"key = {key}")
    print("=" * 80)

    inspect_configuration(
        all_results=all_results,
        key=key,
        label_idx=label_idx,
        label_names=label_names,
    )

    _ = plot_coverage_and_width_per_bin(
        all_results=all_results,
        key=key,
        label_idx=label_idx,
        label_names=label_names,
        plot_labels=plot_labels,
        confidence_level=confidence_level,
        width_stat="median",
        width_space="phys",
        y_std=y_std,
    )

## Final Report

In [ ]:
report_cols = [
    "label",
    "taxonomy_mode",
    "interval_mode",
    "n_bins",
    "global_coverage",
    "min_coverage_per_bin",
    "n_bins_under_2sigma",
    "under_bin_fraction_2sigma",
    "global_median_width_phys",
    "global_tail_miss_imbalance",
    "min_count_per_bin",
    "selection_policy",
]

final_report_df = (
    final_by_label[report_cols]
    .sort_values("label")
    .reset_index(drop=True)
)

final_report_df

In [ ]:
final_selection_dir = mondrian_results_dir / "final_selection"
final_selection_dir.mkdir(parents=True, exist_ok=True)

conservative_path = final_selection_dir / (
    prediction_file.stem
    + f"_mondrian_conservative_configs_{mode}.csv"
)

efficient_path = final_selection_dir / (
    prediction_file.stem
    + f"_mondrian_efficient_configs_{mode}.csv"
)

comparison_path = final_selection_dir / (
    prediction_file.stem
    + f"_mondrian_conservative_vs_efficient_{mode}.csv"
)

conservative_by_label.to_csv(
    conservative_path,
    index=False,
)

efficient_by_label.to_csv(
    efficient_path,
    index=False,
)

selection_comparison_df.to_csv(
    comparison_path,
    index=False,
)

print("Saved conservative selection:", conservative_path)
print("Saved efficient selection:", efficient_path)
print("Saved comparison:", comparison_path)

In [ ]:
for row in final_by_label.itertuples():
    print("=" * 80)
    print(f"Label: {row.label}")
    print(f"Selection policy: {row.selection_policy}")
    print(f"Config: taxonomy={row.taxonomy_mode}, interval={row.interval_mode}, n_bins={row.n_bins}")
    print(f"Global coverage: {row.global_coverage:.4f}")
    print(f"Median physical width: {row.global_median_width_phys:.4f}")
    print(f"n_bad_bins_p005: {row.n_bad_bins_p005}")
    print(f"bad_bin_fraction: {row.bad_bin_fraction:.3f}")
    print(f"n_bins_under_2sigma: {row.n_bins_under_2sigma}")
    print(f"n_bins_outside_2sigma: {row.n_bins_outside_2sigma}")
    print(f"max_undercoverage_gap: {row.max_undercoverage_gap:.4f}")

    if row.selection_policy == "conservative_best_available":
        print(
            "NOTE: No configuration with zero locally undercovered bins "
            "was available for this label. The selected configuration "
            "is the safest conservative fallback."
        )

## (Optional) Inspecting a Configuration

In [ ]:
# Optional manual inspection.
# This does not alter final_by_label.

key = ("difficulty", "asymmetric", 12)
manual_label = "chirp_mass"
manual_label_idx = label_names.index(manual_label)

inspect_configuration(
    all_results=all_results,
    key=key,
    label_idx=manual_label_idx,
    label_names=label_names,
)

In [ ]:
# Checking all good :D

print("Conservative:")
display(conservative_by_label[final_display_cols])

print("Efficient:")
display(efficient_by_label[final_display_cols])

print("Primary final_by_label:")
display(final_by_label[final_display_cols])

In [ ]:
assert final_by_label.equals(conservative_by_label)
assert len(final_by_label) == len(label_names)
assert final_by_label["label"].nunique() == len(label_names)
assert final_by_label["global_within_2sigma"].all()
assert (final_by_label["min_count_per_bin"] >= MIN_COUNT_PER_BIN).all()

The conservative Mondrian selection achieves global coverage compatible with the nominal 90% level and no bins below the local 2σ coverage threshold for all three target parameters. Difficulty-based taxonomies are selected in every case. The final configurations use 4 bins for chirp mass, 6 bins for total mass and 4 bins for effective spin. An efficient sensitivity policy selects the same configurations for the mass parameters, while for effective spin it favors 32 bins and reduces interval width by approximately 6.6%, at the cost of three locally undercovered bins. Therefore, the conservative policy is retained as the primary result.

Hay tres conclusiones interesantes:

1. Difficulty funciona mejor que prediction para los tres parámetros.
2. Más bins no siempre mejoran el resultado; para las masas, pocos bins son suficientes y más estables.
3. chi_eff admite intervalos más estrechos con 32 bins, pero pierde robustez local.

Esto encaja con la idea de que el error de chi_eff es más heterogéneo.

CHI_EFF

En la selección conservadora aparece:

```text
n_bins_under_2sigma = 0
n_bad_bins_p005 = 1
```

Esto no es una contradicción.

Significa que:

- según la banda normal aproximada de 2σ, ningún bin cae por debajo;
- según el test binomial unilateral, un bin tiene evidencia de infracobertura con p < 0.05.

El bin seguramente está muy cerca del límite. Por eso un criterio lo marca y el otro no.

La interpretación correcta es:

la configuración conservadora de chi_eff pasa el criterio principal de 2σ, pero existe una señal débil de infracobertura local en un bin según el test binomial.

TOTAL MASS

Tiene un tail imbalance de aproximadamente 0.028.

Eso significa que los fallos de cobertura están algo más cargados en una cola que en la otra:

- lower miss rate = 0.0347
- upper miss rate = 0.0627

La cobertura global sigue siendo correcta, pero hay cierta asimetría residual.

Como la configuración elegida es simétrica, esto sugiere que una configuración asimétrica podría equilibrar mejor las colas, aunque quizá con mayor anchura o peor validez local.

## $\chi_{\mathrm{eff}}$ study

### Physical-region coverage diagnostic for $\chi_{eff}$

In [ ]:
# ============================================================
# Select the final conservative chi_eff configuration
# ============================================================

chi_final_row = final_by_label.loc[
    final_by_label["label"] == "chi_eff"
].iloc[0]

chi_key = (
    chi_final_row["taxonomy_mode"],
    chi_final_row["interval_mode"],
    int(chi_final_row["n_bins"]),
)

chi_label_idx = int(chi_final_row["label_index"])

assert chi_key in all_results, (
    f"Selected configuration {chi_key} not found in all_results"
)

chi_result = all_results[chi_key]

print("Selected chi_eff configuration:")
print(" taxonomy_mode:", chi_key[0])
print(" interval_mode:", chi_key[1])
print(" n_bins:", chi_key[2])
print(" label_index:", chi_label_idx)

print("\nResult object:")
print(" type:", type(chi_result))
print(" lower shape:", chi_result.lower.shape)
print(" upper shape:", chi_result.upper.shape)
print(" metrics keys:", list(chi_result.metrics.keys()))

In [ ]:
# ============================================================
# Extract chi_eff test intervals
# ============================================================

# result.lower and result.upper are in standardized label space.
chi_lower_std = chi_result.lower[:, chi_label_idx]
chi_upper_std = chi_result.upper[:, chi_label_idx]

chi_true_std = y_test[:, chi_label_idx]
chi_pred_std = pred_test[:, chi_label_idx]

chi_mean = y_mean[chi_label_idx]
chi_scale = y_std[chi_label_idx]

# Convert everything to physical chi_eff units.
chi_true_phys = chi_true_std * chi_scale + chi_mean
chi_pred_phys = chi_pred_std * chi_scale + chi_mean
chi_lower_phys = chi_lower_std * chi_scale + chi_mean
chi_upper_phys = chi_upper_std * chi_scale + chi_mean

chi_covered = (
    (chi_true_phys >= chi_lower_phys)
    & (chi_true_phys <= chi_upper_phys)
)

chi_width_phys = chi_upper_phys - chi_lower_phys
chi_abs_error_phys = np.abs(chi_true_phys - chi_pred_phys)

print("n_test:", len(chi_true_phys))
print("Global coverage:", chi_covered.mean())
print("Median physical width:", np.median(chi_width_phys))
print("Mean physical width:", np.mean(chi_width_phys))
print("Minimum width:", np.min(chi_width_phys))
print("Maximum width:", np.max(chi_width_phys))

assert len(chi_true_phys) == len(chi_lower_phys)
assert np.all(chi_upper_phys >= chi_lower_phys)

In [ ]:
# ============================================================
# Coverage across true chi_eff regions
# ============================================================

chi_edges = np.linspace(-1.0, 1.0, 9)

chi_physical_bin_idx = np.digitize(
    chi_true_phys,
    bins=chi_edges[1:-1],
    right=False,
)

chi_coverage_rows = []

for bin_idx in range(len(chi_edges) - 1):
    mask = chi_physical_bin_idx == bin_idx
    n_bin = int(mask.sum())

    if n_bin == 0:
        continue

    coverage_bin = float(np.mean(chi_covered[mask]))

    sigma_bin = np.sqrt(
        confidence_level
        * (1.0 - confidence_level)
        / n_bin
    )

    lower_2sigma = confidence_level - 2.0 * sigma_bin
    upper_2sigma = confidence_level + 2.0 * sigma_bin

    lower_miss_rate = float(np.mean(
        chi_true_phys[mask] < chi_lower_phys[mask]
    ))

    upper_miss_rate = float(np.mean(
        chi_true_phys[mask] > chi_upper_phys[mask]
    ))

    abs_error_bin = chi_abs_error_phys[mask]

    chi_coverage_rows.append({
        "bin_idx": bin_idx,
        "chi_low": chi_edges[bin_idx],
        "chi_high": chi_edges[bin_idx + 1],
        "chi_center": 0.5 * (
            chi_edges[bin_idx] + chi_edges[bin_idx + 1]
        ),
        "n_samples": n_bin,
        "coverage": coverage_bin,
        "sigma_bin": sigma_bin,
        "lower_2sigma": lower_2sigma,
        "upper_2sigma": upper_2sigma,
        "within_2sigma": (
            lower_2sigma <= coverage_bin <= upper_2sigma
        ),
        "undercovered_2sigma": (
            coverage_bin < lower_2sigma
        ),
        "median_width_phys": float(
            np.median(chi_width_phys[mask])
        ),
        "mean_width_phys": float(
            np.mean(chi_width_phys[mask])
        ),
        "MAE": float(np.mean(abs_error_bin)),
        "q90_abs_error": float(
            np.quantile(abs_error_bin, 0.90)
        ),
        "lower_miss_rate": lower_miss_rate,
        "upper_miss_rate": upper_miss_rate,
        "tail_miss_imbalance": abs(
            lower_miss_rate - upper_miss_rate
        ),
    })

chi_physical_coverage_df = pd.DataFrame(
    chi_coverage_rows
)

chi_physical_coverage_df

#### Coverage plot

In [ ]:
plt.figure(figsize=(8, 4.8))

plt.plot(
    chi_physical_coverage_df["chi_center"],
    chi_physical_coverage_df["coverage"],
    marker="o",
    label="Empirical coverage",
)

plt.fill_between(
    chi_physical_coverage_df["chi_center"],
    chi_physical_coverage_df["lower_2sigma"],
    chi_physical_coverage_df["upper_2sigma"],
    alpha=0.45,
    label="Nominal 90% ± 2σ",
)

plt.axhline(
    confidence_level,
    linestyle="--",
    label="Nominal coverage",
)

plt.xlabel(r"True $\chi_{\mathrm{eff}}$ bin center")
plt.ylabel("Coverage")
plt.title(
    r"Conformal coverage across true $\chi_{\mathrm{eff}}$"
)
plt.ylim(0.75, 1.0)
plt.grid(True)
plt.legend()
plt.show()

#### Width plot

In [ ]:
plt.figure(figsize=(8, 4.8))

plt.plot(
    chi_physical_coverage_df["chi_center"],
    chi_physical_coverage_df["median_width_phys"],
    marker="o",
    label="Median interval width",
)

plt.xlabel(r"True $\chi_{\mathrm{eff}}$ bin center")
plt.ylabel(r"Median interval width")
plt.title(
    r"Interval width across true $\chi_{\mathrm{eff}}$"
)
plt.grid(True)
plt.legend()
plt.show()

#### Local error with width

In [ ]:
chi_error_width_df = chi_physical_coverage_df[
    [
        "chi_center",
        "n_samples",
        "MAE",
        "q90_abs_error",
        "median_width_phys",
        "coverage",
    ]
].copy()

chi_error_width_df["median_half_width"] = (
    chi_error_width_df["median_width_phys"] / 2.0
)

chi_error_width_df

In [ ]:
plt.figure(figsize=(8, 4.8))

plt.plot(
    chi_error_width_df["chi_center"],
    chi_error_width_df["q90_abs_error"],
    marker="o",
    label="q90 absolute error",
)

plt.plot(
    chi_error_width_df["chi_center"],
    chi_error_width_df["median_half_width"],
    marker="s",
    label="Median interval half-width",
)

plt.xlabel(r"True $\chi_{\mathrm{eff}}$ bin center")
plt.ylabel(r"$\chi_{\mathrm{eff}}$ scale")
plt.title(
    r"Prediction error and interval size across true $\chi_{\mathrm{eff}}$"
)
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
region_masks = {
    "negative_extreme": chi_true_phys < -0.5,
    "negative_moderate": (
        (chi_true_phys >= -0.5)
        & (chi_true_phys < -0.2)
    ),
    "central": np.abs(chi_true_phys) <= 0.2,
    "positive_moderate": (
        (chi_true_phys > 0.2)
        & (chi_true_phys <= 0.5)
    ),
    "positive_extreme": chi_true_phys > 0.5,
}

chi_conformal_region_rows = []

for region_name, mask in region_masks.items():
    n_region = int(mask.sum())

    coverage_region = float(
        np.mean(chi_covered[mask])
    )

    sigma_region = np.sqrt(
        confidence_level
        * (1.0 - confidence_level)
        / n_region
    )

    lower_2sigma = (
        confidence_level - 2.0 * sigma_region
    )

    upper_2sigma = (
        confidence_level + 2.0 * sigma_region
    )

    lower_miss_rate = float(np.mean(
        chi_true_phys[mask] < chi_lower_phys[mask]
    ))

    upper_miss_rate = float(np.mean(
        chi_true_phys[mask] > chi_upper_phys[mask]
    ))

    chi_conformal_region_rows.append({
        "region": region_name,
        "n_samples": n_region,
        "coverage": coverage_region,
        "lower_2sigma": lower_2sigma,
        "upper_2sigma": upper_2sigma,
        "within_2sigma": (
            lower_2sigma
            <= coverage_region
            <= upper_2sigma
        ),
        "undercovered_2sigma": (
            coverage_region < lower_2sigma
        ),
        "median_width_phys": float(
            np.median(chi_width_phys[mask])
        ),
        "MAE": float(
            np.mean(chi_abs_error_phys[mask])
        ),
        "q90_abs_error": float(
            np.quantile(
                chi_abs_error_phys[mask],
                0.90,
            )
        ),
        "lower_miss_rate": lower_miss_rate,
        "upper_miss_rate": upper_miss_rate,
        "tail_miss_imbalance": abs(
            lower_miss_rate - upper_miss_rate
        ),
    })

chi_conformal_region_df = pd.DataFrame(
    chi_conformal_region_rows
)

chi_conformal_region_df

#### Results

Although the selected difficulty-based Mondrian configuration achieves global coverage close to the nominal 90% level, coverage is strongly heterogeneous across the true effective-spin range. Coverage falls to approximately 72% and 76% in the most negative and positive bins, respectively, while central regions are overcovered at approximately 93–94%. The miss direction follows the regression-to-the-mean bias of the point predictor: negative extremes miss below the lower interval boundary, whereas positive extremes miss above the upper boundary. Therefore, the difficulty taxonomy does not fully capture the physical error heterogeneity associated with effective-spin magnitude.

### Predicted-$\chi_{eff}$ inspection

¿La predicción puntual de chi_eff, que sí está disponible en inferencia, permite identificar regiones con distinto error y distinta cobertura?

In [ ]:
# ============================================================
# Diagnostic across predicted chi_eff
# ============================================================

pred_chi_edges = np.linspace(-1.0, 1.0, 9)

pred_chi_bin_idx = np.digitize(
    chi_pred_phys,
    bins=pred_chi_edges[1:-1],
    right=False,
)

pred_chi_rows = []

for bin_idx in range(len(pred_chi_edges) - 1):
    mask = pred_chi_bin_idx == bin_idx
    n_bin = int(mask.sum())

    if n_bin == 0:
        continue

    coverage_bin = float(np.mean(chi_covered[mask]))

    sigma_bin = np.sqrt(
        confidence_level
        * (1.0 - confidence_level)
        / n_bin
    )

    lower_2sigma = confidence_level - 2.0 * sigma_bin
    upper_2sigma = confidence_level + 2.0 * sigma_bin

    lower_miss_rate = float(np.mean(
        chi_true_phys[mask] < chi_lower_phys[mask]
    ))

    upper_miss_rate = float(np.mean(
        chi_true_phys[mask] > chi_upper_phys[mask]
    ))

    residual_bin = (
        chi_true_phys[mask] - chi_pred_phys[mask]
    )
    abs_error_bin = np.abs(residual_bin)

    pred_chi_rows.append({
        "bin_idx": bin_idx,
        "pred_chi_low": pred_chi_edges[bin_idx],
        "pred_chi_high": pred_chi_edges[bin_idx + 1],
        "pred_chi_center": 0.5 * (
            pred_chi_edges[bin_idx]
            + pred_chi_edges[bin_idx + 1]
        ),
        "n_samples": n_bin,
        "pred_mean": float(
            np.mean(chi_pred_phys[mask])
        ),
        "true_mean": float(
            np.mean(chi_true_phys[mask])
        ),
        "bias_true_minus_pred": float(
            np.mean(residual_bin)
        ),
        "MAE": float(np.mean(abs_error_bin)),
        "RMSE": float(np.sqrt(
            np.mean(residual_bin**2)
        )),
        "q90_abs_error": float(
            np.quantile(abs_error_bin, 0.90)
        ),
        "coverage": coverage_bin,
        "lower_2sigma": lower_2sigma,
        "upper_2sigma": upper_2sigma,
        "within_2sigma": (
            lower_2sigma <= coverage_bin <= upper_2sigma
        ),
        "undercovered_2sigma": (
            coverage_bin < lower_2sigma
        ),
        "median_width_phys": float(
            np.median(chi_width_phys[mask])
        ),
        "lower_miss_rate": lower_miss_rate,
        "upper_miss_rate": upper_miss_rate,
        "tail_miss_imbalance": abs(
            lower_miss_rate - upper_miss_rate
        ),
    })

pred_chi_diagnostic_df = pd.DataFrame(
    pred_chi_rows
)

pred_chi_diagnostic_df

Qué queremos observar

Si predicted chi_eff es útil como taxonomía, deberíamos ver:

error diferente entre bins;
cobertura diferente entre bins;
sesgo que cambia con la predicción;
intervalos que podrían calibrarse separadamente.

#### Distribución de muestras por predicción

La regresión hacia cero probablemente comprimirá las predicciones, así que quizá los bins extremos estén poco poblados.

Esto importa porque una taxonomía fija basada en intervalos [-1,1] podría producir bins casi vacíos debido a la contracción.

In [ ]:
plt.figure(figsize=(8, 4.5))

plt.bar(
    pred_chi_diagnostic_df["pred_chi_center"],
    pred_chi_diagnostic_df["n_samples"],
    width=np.diff(pred_chi_edges) * 0.85,
)

plt.xlabel(r"Predicted $\chi_{\mathrm{eff}}$ bin center")
plt.ylabel("Number of test samples")
plt.title(
    r"Distribution across predicted $\chi_{\mathrm{eff}}$"
)
plt.grid(True, axis="y")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4.8))

plt.plot(
    pred_chi_diagnostic_df["pred_chi_center"],
    pred_chi_diagnostic_df["coverage"],
    marker="o",
    label="Empirical coverage",
)

plt.fill_between(
    pred_chi_diagnostic_df["pred_chi_center"],
    pred_chi_diagnostic_df["lower_2sigma"],
    pred_chi_diagnostic_df["upper_2sigma"],
    alpha=0.45,
    label="Nominal 90% ± 2σ",
)

plt.axhline(
    confidence_level,
    linestyle="--",
    label="Nominal coverage",
)

plt.xlabel(r"Predicted $\chi_{\mathrm{eff}}$ bin center")
plt.ylabel("Coverage")
plt.title(
    r"Coverage across predicted $\chi_{\mathrm{eff}}$"
)
plt.ylim(0.75, 1.0)
plt.grid(True)
plt.legend()
plt.show()

Interpretación

Si los bins con predicciones muy negativas o muy positivas están infracubiertos, predicted chi_eff está detectando parte del problema y puede ser una buena taxonomía.

Si la cobertura parece uniforme por predicción pero falla por true chi_eff, significa que la regresión hacia cero oculta los extremos físicos y la predicción sola no basta.

In [ ]:
plt.figure(figsize=(8, 4.8))

plt.plot(
    pred_chi_diagnostic_df["pred_chi_center"],
    pred_chi_diagnostic_df["bias_true_minus_pred"],
    marker="o",
    label="Bias: true - predicted",
)

plt.axhline(0.0, linestyle="--")
plt.axvline(0.0, linestyle=":")

plt.xlabel(r"Predicted $\chi_{\mathrm{eff}}$ bin center")
plt.ylabel("Mean residual")
plt.title(
    r"Bias across predicted $\chi_{\mathrm{eff}}$"
)
plt.grid(True)
plt.legend()
plt.show()

Esta tabla es muy útil para medir cuánto “esconde” la contracción.

In [ ]:
pred_true_mapping_df = pred_chi_diagnostic_df[
    [
        "pred_chi_low",
        "pred_chi_high",
        "n_samples",
        "pred_mean",
        "true_mean",
        "bias_true_minus_pred",
        "MAE",
        "q90_abs_error",
        "coverage",
    ]
].copy()

pred_true_mapping_df

In [ ]:
plt.figure(figsize=(6.5, 5.5))

plt.plot(
    pred_true_mapping_df["pred_mean"],
    pred_true_mapping_df["true_mean"],
    marker="o",
    label="Mean true value per prediction bin",
)

lims = [
    min(
        pred_true_mapping_df["pred_mean"].min(),
        pred_true_mapping_df["true_mean"].min(),
    ),
    max(
        pred_true_mapping_df["pred_mean"].max(),
        pred_true_mapping_df["true_mean"].max(),
    ),
]

plt.plot(
    lims,
    lims,
    linestyle="--",
    label="Identity",
)

plt.xlabel(r"Mean predicted $\chi_{\mathrm{eff}}$")
plt.ylabel(r"Mean true $\chi_{\mathrm{eff}}$")
plt.title(
    r"True value associated with predicted $\chi_{\mathrm{eff}}$ bins"
)
plt.grid(True)
plt.legend()
plt.show()

Aquí, si los valores verdaderos son sistemáticamente más extremos que las predicciones:

pred_mean = -0.5 → true_mean ≈ -0.65

pred_mean = +0.5 → true_mean ≈ +0.65

la predicción sí contiene información sobre el sesgo y puede calibrarse por regiones.

#### Qué valores verdaderos hay dentro de cada bin de predicción

No basta con la media. Queremos saber la dispersión del valor verdadero condicionada a la predicción.

Añade cuantiles de true chi_eff por bin de predicción:

In [ ]:
pred_conditional_true_rows = []

for bin_idx in range(len(pred_chi_edges) - 1):
    mask = pred_chi_bin_idx == bin_idx
    n_bin = int(mask.sum())

    if n_bin == 0:
        continue

    true_values = chi_true_phys[mask]
    pred_values = chi_pred_phys[mask]

    pred_conditional_true_rows.append({
        "bin_idx": bin_idx,
        "pred_low": pred_chi_edges[bin_idx],
        "pred_high": pred_chi_edges[bin_idx + 1],
        "n_samples": n_bin,
        "pred_median": float(
            np.median(pred_values)
        ),
        "true_q05": float(
            np.quantile(true_values, 0.05)
        ),
        "true_q25": float(
            np.quantile(true_values, 0.25)
        ),
        "true_median": float(
            np.median(true_values)
        ),
        "true_q75": float(
            np.quantile(true_values, 0.75)
        ),
        "true_q95": float(
            np.quantile(true_values, 0.95)
        ),
    })

pred_conditional_true_df = pd.DataFrame(
    pred_conditional_true_rows
)

pred_conditional_true_df

In [ ]:
plt.figure(figsize=(8, 5))

x = pred_conditional_true_df["pred_median"]

plt.plot(
    x,
    pred_conditional_true_df["true_median"],
    marker="o",
    label="Median true chi_eff",
)

plt.fill_between(
    x,
    pred_conditional_true_df["true_q25"],
    pred_conditional_true_df["true_q75"],
    alpha=0.25,
    label="True IQR",
)

plt.fill_between(
    x,
    pred_conditional_true_df["true_q05"],
    pred_conditional_true_df["true_q95"],
    alpha=0.12,
    label="True 5–95%",
)

plt.plot(
    x,
    x,
    linestyle="--",
    label="Prediction = truth",
)

plt.xlabel(r"Median predicted $\chi_{\mathrm{eff}}$")
plt.ylabel(r"Conditional true $\chi_{\mathrm{eff}}$")
plt.title(
    r"True $\chi_{\mathrm{eff}}$ distribution conditioned on prediction"
)
plt.grid(True)
plt.legend()
plt.show()

Esto muestra si un mismo valor predicho mezcla valores verdaderos muy distintos. Si la banda 5–95% es muy ancha, predicted chi_eff por sí solo no separa bien dificultad.

#### Dónde terminan los extremos verdaderos en el espacio predicho

Ahora queremos saber qué predicciones produce el modelo cuando el valor real es extremo.

In [ ]:
true_region_masks = {
    "true_negative_extreme": chi_true_phys < -0.5,
    "true_central": np.abs(chi_true_phys) <= 0.2,
    "true_positive_extreme": chi_true_phys > 0.5,
}

true_region_prediction_rows = []

for region_name, mask in true_region_masks.items():
    pred_values = chi_pred_phys[mask]

    true_region_prediction_rows.append({
        "true_region": region_name,
        "n_samples": int(mask.sum()),
        "pred_mean": float(
            np.mean(pred_values)
        ),
        "pred_std": float(
            np.std(pred_values)
        ),
        "pred_q05": float(
            np.quantile(pred_values, 0.05)
        ),
        "pred_q25": float(
            np.quantile(pred_values, 0.25)
        ),
        "pred_median": float(
            np.median(pred_values)
        ),
        "pred_q75": float(
            np.quantile(pred_values, 0.75)
        ),
        "pred_q95": float(
            np.quantile(pred_values, 0.95)
        ),
    })

true_region_prediction_df = pd.DataFrame(
    true_region_prediction_rows
)

true_region_prediction_df

Esto permite responder:

¿Los verdaderos extremos quedan suficientemente separados en el espacio de predicción como para identificarlos?

Por ejemplo, si casi todos los verdaderos chi_eff > 0.5 tienen predicción > 0.3, podrías usar la predicción como taxonomía.

Si muchos extremos se predicen cerca de cero, no bastará.

#### Detección de extremos mediante la predicción

Podemos tratarlo como una clasificación diagnóstica simple.

In [ ]:
threshold_grid = np.linspace(0.0, 0.8, 41)

extreme_detection_rows = []

true_positive_extreme = chi_true_phys > 0.5
true_negative_extreme = chi_true_phys < -0.5

for threshold in threshold_grid:
    predicted_positive_extreme = (
        chi_pred_phys > threshold
    )
    predicted_negative_extreme = (
        chi_pred_phys < -threshold
    )

    pos_recall = (
        np.mean(
            predicted_positive_extreme[
                true_positive_extreme
            ]
        )
        if true_positive_extreme.any()
        else np.nan
    )

    neg_recall = (
        np.mean(
            predicted_negative_extreme[
                true_negative_extreme
            ]
        )
        if true_negative_extreme.any()
        else np.nan
    )

    pos_precision = (
        np.mean(
            true_positive_extreme[
                predicted_positive_extreme
            ]
        )
        if predicted_positive_extreme.any()
        else np.nan
    )

    neg_precision = (
        np.mean(
            true_negative_extreme[
                predicted_negative_extreme
            ]
        )
        if predicted_negative_extreme.any()
        else np.nan
    )

    extreme_detection_rows.append({
        "threshold": threshold,
        "positive_recall": pos_recall,
        "positive_precision": pos_precision,
        "negative_recall": neg_recall,
        "negative_precision": neg_precision,
    })

extreme_detection_df = pd.DataFrame(
    extreme_detection_rows
)

extreme_detection_df

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    extreme_detection_df["threshold"],
    extreme_detection_df["positive_recall"],
    label="Positive extreme recall",
)

plt.plot(
    extreme_detection_df["threshold"],
    extreme_detection_df["positive_precision"],
    label="Positive extreme precision",
)

plt.plot(
    extreme_detection_df["threshold"],
    extreme_detection_df["negative_recall"],
    linestyle="--",
    label="Negative extreme recall",
)

plt.plot(
    extreme_detection_df["threshold"],
    extreme_detection_df["negative_precision"],
    linestyle="--",
    label="Negative extreme precision",
)

plt.xlabel(r"Threshold on predicted $|\chi_{\mathrm{eff}}|$")
plt.ylabel("Score")
plt.title(
    r"Detection of true extreme $\chi_{\mathrm{eff}}$ from predictions"
)
plt.grid(True)
plt.legend()
plt.show()

La predicción de chi_eff contiene información operacional útil, pero la heterogeneidad de los valores verdaderos condicionados a predicciones centrales impide que sea una taxonomía suficiente por sí sola. La siguiente línea de trabajo adecuada es una taxonomía híbrida que combine bins por cuantiles de predicted chi_eff con bins de dificultad.

predicted chi_eff es una señal útil pero incompleta. Tiene suficiente información para distinguir regiones de comportamiento diferente, pero la regresión hacia la media hace que muchos valores verdaderos extremos aparezcan como predicciones centrales. El recall mide cuántos extremos reales detectamos; la precision mide cuántos de los casos marcados como extremos lo son realmente. Los resultados moderados de ambas métricas y la amplia distribución de valores verdaderos condicionada a predicciones centrales justifican añadir una segunda variable, la dificultad, para separar casos fáciles y difíciles dentro de la misma región predicha.

### Hybrid Mondrian taxonomy for chi_eff

In [ ]:
from src.conformal.hybrid import run_hybrid_mondrian_regression

print("Hybrid import OK")

In [ ]:
# ============================================================
# Prepare chi_eff-only arrays
# ============================================================

chi_label = "chi_eff"
chi_idx = label_names.index(chi_label)

pred_cal_chi = pred_cal[:, [chi_idx]]
pred_test_chi = pred_test[:, [chi_idx]]

y_cal_chi = y_cal[:, [chi_idx]]
y_test_chi = y_test[:, [chi_idx]]

print("pred_cal_chi:", pred_cal_chi.shape)
print("pred_test_chi:", pred_test_chi.shape)
print("y_cal_chi:", y_cal_chi.shape)
print("y_test_chi:", y_test_chi.shape)
print("emb_cal:", emb_cal.shape)
print("emb_test:", emb_test.shape)

Test configuration:

- prediction bins = 3
- difficulty bins = 3
- interval mode = asymmetric

In [ ]:
hybrid_result_3x3 = run_hybrid_mondrian_regression(
    pred_cal=pred_cal_chi,
    pred_test=pred_test_chi,
    y_cal=y_cal_chi,
    y_test=y_test_chi,
    cal_embedding=emb_cal,
    target_embedding=emb_test,
    n_prediction_bins=3,
    n_difficulty_bins=3,
    n_neighbors=5,
    confidence_level=0.90,
    apply_jitter=True,
    jitter_variation=1e-10,
    interval_mode="asymmetric",
    min_samples_per_bin=10,
    tolerance_sigmas=(1, 2, 3),
    random_seed=123,
    standardize_embeddings=False,
)

print("Hybrid 3x3 completed")
print("Joint bins:", hybrid_result_3x3.n_joint_bins)

In [ ]:
metrics_3x3 = hybrid_result_3x3.metrics

print("Global coverage:",
      metrics_3x3["global_coverage"][0])

print("Median width standardized:",
      metrics_3x3["global_median_width"][0])

print("Counts per joint bin:")
print(metrics_3x3["counts_per_bin"][:, 0])

print("Coverage per joint bin:")
print(metrics_3x3["coverage_per_bin"][:, 0])


assert hybrid_result_3x3.lower.shape == y_test_chi.shape
assert hybrid_result_3x3.upper.shape == y_test_chi.shape
assert hybrid_result_3x3.bin_indices_test.shape == y_test_chi.shape
assert hybrid_result_3x3.n_joint_bins == 9

In [ ]:
hybrid_group_rows = []

for pred_bin in range(hybrid_result_3x3.n_prediction_bins):
    for diff_bin in range(hybrid_result_3x3.n_difficulty_bins):
        joint_bin = (
            pred_bin * hybrid_result_3x3.n_difficulty_bins
            + diff_bin
        )

        test_mask = (
            hybrid_result_3x3.bin_indices_test[:, 0]
            == joint_bin
        )

        cal_mask = (
            hybrid_result_3x3.bin_indices_cal[:, 0]
            == joint_bin
        )

        hybrid_group_rows.append({
            "joint_bin": joint_bin,
            "prediction_bin": pred_bin,
            "difficulty_bin": diff_bin,
            "n_cal": int(cal_mask.sum()),
            "n_test": int(test_mask.sum()),
            "test_coverage": (
                metrics_3x3["coverage_per_bin"][
                    joint_bin, 0
                ]
            ),
            "median_width_std": (
                metrics_3x3["median_width_per_bin"][
                    joint_bin, 0
                ]
            ),
            "lower_miss_rate": (
                metrics_3x3["lower_miss_rate_per_bin"][
                    joint_bin, 0
                ]
            ),
            "upper_miss_rate": (
                metrics_3x3["upper_miss_rate_per_bin"][
                    joint_bin, 0
                ]
            ),
        })

hybrid_group_df = pd.DataFrame(hybrid_group_rows)
hybrid_group_df

- prediction_bin = 0 → predicciones bajas
- prediction_bin = 1 → predicciones centrales
- prediction_bin = 2 → predicciones altas

- difficulty_bin = 0 → dificultad baja
- difficulty_bin = 1 → dificultad media
- difficulty_bin = 2 → dificultad alta

Como son cuantiles, cada componente individual tiende a estar balanceado, aunque los grupos conjuntos pueden no tener exactamente el mismo tamaño porque prediction y difficulty están correlacionados.



In [ ]:
chi_scale = y_std[chi_idx]

hybrid_group_df["median_width_phys"] = (
    hybrid_group_df["median_width_std"]
    * chi_scale
)

global_median_width_phys_3x3 = (
    metrics_3x3["global_median_width"][0]
    * chi_scale
)

print(
    "Global median physical width:",
    global_median_width_phys_3x3,
)

In [ ]:
chi_true_phys_hybrid = (
    y_test_chi[:, 0] * chi_scale
    + y_mean[chi_idx]
)

chi_lower_phys_hybrid = (
    hybrid_result_3x3.lower[:, 0] * chi_scale
    + y_mean[chi_idx]
)

chi_upper_phys_hybrid = (
    hybrid_result_3x3.upper[:, 0] * chi_scale
    + y_mean[chi_idx]
)

chi_width_phys_hybrid = (
    chi_upper_phys_hybrid
    - chi_lower_phys_hybrid
)

chi_covered_hybrid = (
    (chi_true_phys_hybrid >= chi_lower_phys_hybrid)
    & (chi_true_phys_hybrid <= chi_upper_phys_hybrid)
)

In [ ]:
chi_edges = np.linspace(-1.0, 1.0, 9)

chi_true_bin_idx_hybrid = np.digitize(
    chi_true_phys_hybrid,
    bins=chi_edges[1:-1],
    right=False,
)

hybrid_physical_rows = []

for bin_idx in range(len(chi_edges) - 1):
    mask = chi_true_bin_idx_hybrid == bin_idx
    n_bin = int(mask.sum())

    if n_bin == 0:
        continue

    coverage = float(
        chi_covered_hybrid[mask].mean()
    )

    sigma = np.sqrt(
        confidence_level
        * (1.0 - confidence_level)
        / n_bin
    )

    hybrid_physical_rows.append({
        "bin_idx": bin_idx,
        "chi_low": chi_edges[bin_idx],
        "chi_high": chi_edges[bin_idx + 1],
        "chi_center": 0.5 * (
            chi_edges[bin_idx]
            + chi_edges[bin_idx + 1]
        ),
        "n_samples": n_bin,
        "coverage": coverage,
        "lower_2sigma": (
            confidence_level - 2.0 * sigma
        ),
        "upper_2sigma": (
            confidence_level + 2.0 * sigma
        ),
        "undercovered_2sigma": (
            coverage
            < confidence_level - 2.0 * sigma
        ),
        "median_width_phys": float(
            np.median(
                chi_width_phys_hybrid[mask]
            )
        ),
        "lower_miss_rate": float(
            np.mean(
                chi_true_phys_hybrid[mask]
                < chi_lower_phys_hybrid[mask]
            )
        ),
        "upper_miss_rate": float(
            np.mean(
                chi_true_phys_hybrid[mask]
                > chi_upper_phys_hybrid[mask]
            )
        ),
    })

hybrid_physical_coverage_df = pd.DataFrame(
    hybrid_physical_rows
)

hybrid_physical_coverage_df

Compare with conservative baseline

In [ ]:
baseline_physical_compare = (
    chi_physical_coverage_df[
        [
            "chi_center",
            "coverage",
            "median_width_phys",
            "lower_miss_rate",
            "upper_miss_rate",
        ]
    ]
    .rename(columns={
        "coverage": "coverage_baseline",
        "median_width_phys": "width_baseline",
        "lower_miss_rate": "lower_miss_baseline",
        "upper_miss_rate": "upper_miss_baseline",
    })
)

hybrid_physical_compare = (
    hybrid_physical_coverage_df[
        [
            "chi_center",
            "coverage",
            "median_width_phys",
            "lower_miss_rate",
            "upper_miss_rate",
        ]
    ]
    .rename(columns={
        "coverage": "coverage_hybrid_3x3",
        "median_width_phys": "width_hybrid_3x3",
        "lower_miss_rate": "lower_miss_hybrid_3x3",
        "upper_miss_rate": "upper_miss_hybrid_3x3",
    })
)

hybrid_vs_baseline_df = baseline_physical_compare.merge(
    hybrid_physical_compare,
    on="chi_center",
    how="inner",
)

hybrid_vs_baseline_df["coverage_delta"] = (
    hybrid_vs_baseline_df["coverage_hybrid_3x3"]
    - hybrid_vs_baseline_df["coverage_baseline"]
)

hybrid_vs_baseline_df["width_delta"] = (
    hybrid_vs_baseline_df["width_hybrid_3x3"]
    - hybrid_vs_baseline_df["width_baseline"]
)

hybrid_vs_baseline_df

The 3×3 hybrid prediction–difficulty taxonomy achieves stable coverage across its joint Mondrian groups and slightly reduces the global median interval width. However, it does not improve physical conditional coverage in the extreme true effective-spin regions. Coverage decreases from 0.718 to 0.687 in the most negative bin and from 0.762 to 0.728 in the most positive bin. The remaining failures are strongly directional, indicating that the dominant issue is regression-to-the-mean bias rather than insufficient uncertainty stratification alone.

#### With a small grid

In [ ]:
from src.conformal.apply import apply_indices
from src.conformal.binning import BinGrouper
from src.conformal.calibration import ConformalIntervalCalibrator
from src.conformal.difficulty import DifficultyEstimator
from src.conformal.hybrid import HybridQuantileBinner
from src.conformal.metrics import CoverageEvaluator

In [ ]:
# ============================================================
# Precompute chi_eff difficulty scores once
# ============================================================

chi_residuals_cal = y_cal_chi - pred_cal_chi

hybrid_difficulty_model = DifficultyEstimator(
    n_neighbors=5,
    distance_weighted=True,
    standardize_embeddings=False,
)

hybrid_difficulty_model.calibrate_estimator(
    cal_embedding=emb_cal,
    cal_residuals=chi_residuals_cal,
)

difficulty_cal_chi = (
    hybrid_difficulty_model.compute_calibration_difficulty()
)

difficulty_test_chi = (
    hybrid_difficulty_model.compute_target_difficulty(
        target_embedding=emb_test,
    )
)

print("difficulty_cal_chi:", difficulty_cal_chi.shape)
print("difficulty_test_chi:", difficulty_test_chi.shape)

print(
    "Calibration difficulty range:",
    difficulty_cal_chi.min(),
    difficulty_cal_chi.max(),
)

print(
    "Test difficulty range:",
    difficulty_test_chi.min(),
    difficulty_test_chi.max(),
)

In [ ]:
# ============================================================
# Run one hybrid configuration from cached difficulty scores
# ============================================================

def run_cached_hybrid_configuration(
    n_prediction_bins,
    n_difficulty_bins,
    interval_mode,
    random_seed=123,
):
    n_joint_bins = (
        n_prediction_bins * n_difficulty_bins
    )

    hybrid_binner = HybridQuantileBinner(
        n_prediction_bins=n_prediction_bins,
        n_difficulty_bins=n_difficulty_bins,
        apply_jitter=True,
        jitter_variation=1e-10,
        rng=np.random.default_rng(random_seed),
    )

    (
        bin_indices_cal,
        prediction_indices_cal,
        difficulty_indices_cal,
    ) = hybrid_binner.fit_transform(
        prediction_scores_cal=pred_cal_chi,
        difficulty_scores_cal=difficulty_cal_chi,
    )

    (
        prediction_indices_test,
        difficulty_indices_test,
    ) = hybrid_binner.get_component_indices(
        prediction_scores=pred_test_chi,
        difficulty_scores=difficulty_test_chi,
    )

    bin_indices_test = (
        prediction_indices_test * n_difficulty_bins
        + difficulty_indices_test
    ).astype(int)

    grouper = BinGrouper()

    grouped_residuals = grouper.group_by_bin(
        residuals=chi_residuals_cal,
        bin_indices=bin_indices_cal,
        n_bins=n_joint_bins,
    )

    calibrator = ConformalIntervalCalibrator(
        confidence_level=confidence_level,
        interval_mode=interval_mode,
        min_samples_per_bin=10,
    )

    calibrator.fit(grouped_residuals)

    lower, upper = apply_indices(
        values=pred_test_chi,
        bin_indices=bin_indices_test,
        intervals=calibrator.intervals_,
    )

    evaluator = CoverageEvaluator(
        confidence_level=confidence_level,
        tolerance_sigmas=(1, 2, 3),
    )

    metrics = evaluator.evaluate_intervals(
        y=y_test_chi,
        lower_bound=lower,
        upper_bound=upper,
        bin_indices=bin_indices_test,
        n_bins=n_joint_bins,
    )

    return {
        "n_prediction_bins": n_prediction_bins,
        "n_difficulty_bins": n_difficulty_bins,
        "n_joint_bins": n_joint_bins,
        "interval_mode": interval_mode,
        "bin_indices_cal": bin_indices_cal,
        "bin_indices_test": bin_indices_test,
        "prediction_indices_cal": prediction_indices_cal,
        "prediction_indices_test": prediction_indices_test,
        "difficulty_indices_cal": difficulty_indices_cal,
        "difficulty_indices_test": difficulty_indices_test,
        "lower": lower,
        "upper": upper,
        "intervals": calibrator.intervals_,
        "metrics": metrics,
        "hybrid_binner": hybrid_binner,
        "calibrator": calibrator,
    }

In [ ]:
# ============================================================
# Small hybrid grid
# ============================================================

hybrid_bin_grid = [
    (3, 3),
    (4, 3),
    (3, 4),
    (4, 4),
]

hybrid_interval_modes = [
    "symmetric",
    "asymmetric",
]

hybrid_grid_results = {}

for n_pred_bins, n_diff_bins in hybrid_bin_grid:
    for interval_mode in hybrid_interval_modes:

        key = (
            n_pred_bins,
            n_diff_bins,
            interval_mode,
        )

        print("Running:", key)

        hybrid_grid_results[key] = (
            run_cached_hybrid_configuration(
                n_prediction_bins=n_pred_bins,
                n_difficulty_bins=n_diff_bins,
                interval_mode=interval_mode,
                random_seed=123,
            )
        )

print(
    "Completed configurations:",
    len(hybrid_grid_results),
)

In [ ]:
# ============================================================
# Evaluate one result across true chi_eff bins
# ============================================================

def evaluate_hybrid_physical_regions(result):
    lower_phys = (
        result["lower"][:, 0] * chi_scale
        + y_mean[chi_idx]
    )

    upper_phys = (
        result["upper"][:, 0] * chi_scale
        + y_mean[chi_idx]
    )

    covered = (
        (chi_true_phys_hybrid >= lower_phys)
        & (chi_true_phys_hybrid <= upper_phys)
    )

    width_phys = upper_phys - lower_phys

    physical_rows = []

    for bin_idx in range(len(chi_edges) - 1):
        mask = chi_true_bin_idx_hybrid == bin_idx
        n_bin = int(mask.sum())

        coverage = float(covered[mask].mean())

        sigma = np.sqrt(
            confidence_level
            * (1.0 - confidence_level)
            / n_bin
        )

        lower_miss_rate = float(np.mean(
            chi_true_phys_hybrid[mask]
            < lower_phys[mask]
        ))

        upper_miss_rate = float(np.mean(
            chi_true_phys_hybrid[mask]
            > upper_phys[mask]
        ))

        physical_rows.append({
            "bin_idx": bin_idx,
            "chi_low": chi_edges[bin_idx],
            "chi_high": chi_edges[bin_idx + 1],
            "chi_center": 0.5 * (
                chi_edges[bin_idx]
                + chi_edges[bin_idx + 1]
            ),
            "n_samples": n_bin,
            "coverage": coverage,
            "lower_2sigma": (
                confidence_level - 2.0 * sigma
            ),
            "upper_2sigma": (
                confidence_level + 2.0 * sigma
            ),
            "undercovered_2sigma": (
                coverage
                < confidence_level - 2.0 * sigma
            ),
            "median_width_phys": float(
                np.median(width_phys[mask])
            ),
            "lower_miss_rate": lower_miss_rate,
            "upper_miss_rate": upper_miss_rate,
            "tail_miss_imbalance": abs(
                lower_miss_rate
                - upper_miss_rate
            ),
        })

    return pd.DataFrame(physical_rows)

In [ ]:
# ============================================================
# Summarize global, joint-bin and physical coverage
# ============================================================

hybrid_grid_summary_rows = []
hybrid_grid_physical_tables = {}

n_test_chi = len(y_test_chi)

global_sigma = np.sqrt(
    confidence_level
    * (1.0 - confidence_level)
    / n_test_chi
)

global_lower_2sigma = (
    confidence_level - 2.0 * global_sigma
)

global_upper_2sigma = (
    confidence_level + 2.0 * global_sigma
)

for key, result in hybrid_grid_results.items():
    n_pred_bins, n_diff_bins, interval_mode = key

    metrics = result["metrics"]

    coverage_per_joint_bin = (
        metrics["coverage_per_bin"][:, 0]
    )

    counts_per_joint_bin = (
        metrics["counts_per_bin"][:, 0]
    )

    joint_lower_2sigma = (
        metrics["bin_tolerance_normal"][
            "2sigma_low"
        ][:, 0]
    )

    joint_undercovered = (
        coverage_per_joint_bin
        < joint_lower_2sigma
    )

    physical_df = evaluate_hybrid_physical_regions(
        result
    )

    hybrid_grid_physical_tables[key] = physical_df

    negative_extreme_coverage = float(
        physical_df.loc[
            physical_df["bin_idx"] == 0,
            "coverage",
        ].iloc[0]
    )

    positive_extreme_coverage = float(
        physical_df.loc[
            physical_df["bin_idx"] == 7,
            "coverage",
        ].iloc[0]
    )

    physical_undercoverage_count = int(
        physical_df[
            "undercovered_2sigma"
        ].sum()
    )

    global_coverage = float(
        metrics["global_coverage"][0]
    )

    global_median_width_phys = float(
        metrics["global_median_width"][0]
        * chi_scale
    )

    hybrid_grid_summary_rows.append({
        "n_prediction_bins": n_pred_bins,
        "n_difficulty_bins": n_diff_bins,
        "n_joint_bins": (
            n_pred_bins * n_diff_bins
        ),
        "interval_mode": interval_mode,

        "global_coverage": global_coverage,
        "global_within_2sigma": (
            global_lower_2sigma
            <= global_coverage
            <= global_upper_2sigma
        ),
        "global_median_width_phys": (
            global_median_width_phys
        ),

        "min_joint_coverage": float(
            np.nanmin(coverage_per_joint_bin)
        ),
        "n_joint_undercovered_2sigma": int(
            np.sum(joint_undercovered)
        ),
        "min_joint_count_test": int(
            np.min(counts_per_joint_bin)
        ),

        "negative_extreme_coverage": (
            negative_extreme_coverage
        ),
        "positive_extreme_coverage": (
            positive_extreme_coverage
        ),
        "mean_extreme_coverage": (
            0.5
            * (
                negative_extreme_coverage
                + positive_extreme_coverage
            )
        ),
        "worst_extreme_coverage": min(
            negative_extreme_coverage,
            positive_extreme_coverage,
        ),

        "min_physical_coverage": float(
            physical_df["coverage"].min()
        ),
        "n_physical_undercovered_2sigma": (
            physical_undercoverage_count
        ),
        "max_physical_tail_imbalance": float(
            physical_df[
                "tail_miss_imbalance"
            ].max()
        ),
    })

hybrid_grid_summary_df = pd.DataFrame(
    hybrid_grid_summary_rows
)

hybrid_grid_summary_df

In [ ]:
# ============================================================
# Baseline reference
# ============================================================

baseline_global_coverage = float(
    chi_covered.mean()
)

baseline_global_width = float(
    np.median(chi_width_phys)
)

baseline_negative_extreme_coverage = float(
    chi_physical_coverage_df.loc[
        chi_physical_coverage_df["bin_idx"] == 0,
        "coverage",
    ].iloc[0]
)

baseline_positive_extreme_coverage = float(
    chi_physical_coverage_df.loc[
        chi_physical_coverage_df["bin_idx"] == 7,
        "coverage",
    ].iloc[0]
)

baseline_reference_df = pd.DataFrame([{
    "configuration": "difficulty_asymmetric_4_baseline",
    "global_coverage": baseline_global_coverage,
    "global_median_width_phys": baseline_global_width,
    "negative_extreme_coverage": (
        baseline_negative_extreme_coverage
    ),
    "positive_extreme_coverage": (
        baseline_positive_extreme_coverage
    ),
    "mean_extreme_coverage": 0.5 * (
        baseline_negative_extreme_coverage
        + baseline_positive_extreme_coverage
    ),
    "worst_extreme_coverage": min(
        baseline_negative_extreme_coverage,
        baseline_positive_extreme_coverage,
    ),
}])

baseline_reference_df

In [ ]:
eligible_hybrid_df = hybrid_grid_summary_df.loc[
    hybrid_grid_summary_df[
        "global_within_2sigma"
    ]
    & (
        hybrid_grid_summary_df[
            "min_joint_count_test"
        ] >= 200
    )
].copy()

hybrid_grid_ranked_df = (
    eligible_hybrid_df
    .sort_values(
        by=[
            "worst_extreme_coverage",
            "n_physical_undercovered_2sigma",
            "global_median_width_phys",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

hybrid_grid_ranked_df

In [ ]:
hybrid_grid_ranked_df[
    "delta_negative_extreme"
] = (
    hybrid_grid_ranked_df[
        "negative_extreme_coverage"
    ]
    - baseline_negative_extreme_coverage
)

hybrid_grid_ranked_df[
    "delta_positive_extreme"
] = (
    hybrid_grid_ranked_df[
        "positive_extreme_coverage"
    ]
    - baseline_positive_extreme_coverage
)

hybrid_grid_ranked_df[
    "delta_worst_extreme"
] = (
    hybrid_grid_ranked_df[
        "worst_extreme_coverage"
    ]
    - min(
        baseline_negative_extreme_coverage,
        baseline_positive_extreme_coverage,
    )
)

hybrid_grid_ranked_df[
    "delta_width"
] = (
    hybrid_grid_ranked_df[
        "global_median_width_phys"
    ]
    - baseline_global_width
)

hybrid_grid_ranked_df[
    [
        "n_prediction_bins",
        "n_difficulty_bins",
        "interval_mode",
        "global_coverage",
        "global_median_width_phys",
        "min_joint_coverage",
        "n_joint_undercovered_2sigma",
        "negative_extreme_coverage",
        "positive_extreme_coverage",
        "worst_extreme_coverage",
        "n_physical_undercovered_2sigma",
        "delta_negative_extreme",
        "delta_positive_extreme",
        "delta_worst_extreme",
        "delta_width",
    ]
]

In [ ]:
top_hybrid_keys = []

for _, row in hybrid_grid_ranked_df.head(3).iterrows():
    top_hybrid_keys.append((
        int(row["n_prediction_bins"]),
        int(row["n_difficulty_bins"]),
        row["interval_mode"],
    ))

In [ ]:
plt.figure(figsize=(9, 5.5))

plt.plot(
    chi_physical_coverage_df["chi_center"],
    chi_physical_coverage_df["coverage"],
    marker="o",
    linewidth=2,
    label="Baseline difficulty-asymmetric-4",
)

for key in top_hybrid_keys:
    physical_df = hybrid_grid_physical_tables[key]

    label = (
        f"Hybrid {key[0]}x{key[1]} "
        f"{key[2]}"
    )

    plt.plot(
        physical_df["chi_center"],
        physical_df["coverage"],
        marker="o",
        label=label,
    )

plt.axhline(
    confidence_level,
    linestyle="--",
    label="Nominal 90%",
)

plt.xlabel(r"True $\chi_{\mathrm{eff}}$ bin center")
plt.ylabel("Coverage")
plt.title(
    r"Physical coverage: baseline vs hybrid taxonomies"
)
plt.ylim(0.65, 1.0)
plt.grid(True)
plt.legend()
plt.show()

La taxonomía híbrida mejora eficiencia global y validez dentro de sus grupos operacionales, pero no corrige la infracobertura condicionada por el verdadero spin efectivo.

The hybrid prediction–difficulty grid confirms that operational Mondrian groups can be calibrated reliably and can reduce median interval width by up to approximately 9%. However, none of the tested configurations improves conditional coverage in the extreme true effective-spin regions. The best hybrid configuration, 3×4 symmetric, achieves a median width of 0.599 but yields extreme coverages of 0.714 and 0.735, both below the original difficulty-based baseline. Increasing taxonomy resolution further reduces interval width while degrading extreme coverage. These results indicate that the dominant limitation is the regression-to-the-mean bias of the point predictor rather than insufficient binning resolution.

### Linear recalibration of the $\chi_{eff}$ point predictor

In [ ]:
# ============================================================
# Prepare chi_eff arrays for linear recalibration
# ============================================================

chi_label = "chi_eff"
chi_idx = label_names.index(chi_label)

pred_val = data["pred_val"]
y_val = data["y_val"]

pred_val_chi = pred_val[:, chi_idx]
y_val_chi = y_val[:, chi_idx]

pred_cal_chi_1d = pred_cal[:, chi_idx]
y_cal_chi_1d = y_cal[:, chi_idx]

pred_test_chi_1d = pred_test[:, chi_idx]
y_test_chi_1d = y_test[:, chi_idx]

print("val:", pred_val_chi.shape, y_val_chi.shape)
print("cal:", pred_cal_chi_1d.shape, y_cal_chi_1d.shape)
print("test:", pred_test_chi_1d.shape, y_test_chi_1d.shape)

Fit: corrected_prediction = intercept + slope × original_prediction

In [ ]:
# ============================================================
# Fit shrinkage relation using validation only
# ============================================================

shrinkage_slope, shrinkage_intercept = np.polyfit(
    y_val_chi,
    pred_val_chi,
    deg=1,
)

print(
    "Original predictor relation on validation:\n"
    f"predicted = {shrinkage_slope:.6f} * true "
    f"+ {shrinkage_intercept:.6f}"
)

if shrinkage_slope <= 0:
    raise ValueError(
        "The estimated shrinkage slope must be positive."
    )

In [ ]:
# ============================================================
# Invert the validation-estimated shrinkage relation
# ============================================================

pred_val_chi_corrected = (
    pred_val_chi - shrinkage_intercept
) / shrinkage_slope

pred_cal_chi_corrected = (
    pred_cal_chi_1d - shrinkage_intercept
) / shrinkage_slope

pred_test_chi_corrected = (
    pred_test_chi_1d - shrinkage_intercept
) / shrinkage_slope

In [ ]:
# ============================================================
# Diagnostic slopes before and after recalibration
# ============================================================

slope_before, intercept_before = np.polyfit(
    y_test_chi_1d,
    pred_test_chi_1d,
    deg=1,
)

slope_after, intercept_after = np.polyfit(
    y_test_chi_1d,
    pred_test_chi_corrected,
    deg=1,
)

print("Before correction:")
print(
    f"  pred = {slope_before:.6f} * true "
    f"+ {intercept_before:.6f}"
)

print("\nAfter inverse correction:")
print(
    f"  pred = {slope_after:.6f} * true "
    f"+ {intercept_after:.6f}"
)

La transformación divide por aproximadamente 0.77, de modo que amplifica las predicciones y también parte del ruido:

factor de expansión ≈ 1 / 0.77 ≈ 1.29

Por tanto, puede ocurrir que:

disminuya el sesgo en los extremos;
aumente la varianza;
aumente MAE o RMSE en la región central;
aparezcan más predicciones fuera de [-1,1].

Eso no significa automáticamente que sea mala. Tenemos que comparar:

pendiente y bias regional;
MAE/RMSE;
errores extremos;
proporción fuera del rango;
cobertura conformal posterior.

In [ ]:
# ============================================================
# Point-prediction metrics before and after recalibration
# ============================================================

def regression_metrics_1d(y_true, y_pred):
    residual = y_true - y_pred
    abs_error = np.abs(residual)

    ss_res = np.sum(residual**2)
    ss_tot = np.sum(
        (y_true - np.mean(y_true))**2
    )

    return {
        "MSE": float(np.mean(residual**2)),
        "RMSE": float(np.sqrt(np.mean(residual**2))),
        "MAE": float(np.mean(abs_error)),
        "bias_true_minus_pred": float(np.mean(residual)),
        "q50_abs_error": float(np.quantile(abs_error, 0.50)),
        "q90_abs_error": float(np.quantile(abs_error, 0.90)),
        "q95_abs_error": float(np.quantile(abs_error, 0.95)),
        "R2": float(1.0 - ss_res / ss_tot),
    }


metrics_before_std = regression_metrics_1d(
    y_test_chi_1d,
    pred_test_chi_1d,
)

metrics_after_std = regression_metrics_1d(
    y_test_chi_1d,
    pred_test_chi_corrected,
)

point_recalibration_df = pd.DataFrame([
    {
        "model": "original",
        **metrics_before_std,
    },
    {
        "model": "linear_recalibrated",
        **metrics_after_std,
    },
])

point_recalibration_df

In [ ]:
# ============================================================
# Physical-space predictions and residuals
# ============================================================

chi_mean = y_mean[chi_idx]
chi_scale = y_std[chi_idx]

chi_true_test_phys = (
    y_test_chi_1d * chi_scale + chi_mean
)

chi_pred_test_phys_original = (
    pred_test_chi_1d * chi_scale + chi_mean
)

chi_pred_test_phys_corrected = (
    pred_test_chi_corrected * chi_scale + chi_mean
)

chi_residual_original_phys = (
    chi_true_test_phys
    - chi_pred_test_phys_original
)

chi_residual_corrected_phys = (
    chi_true_test_phys
    - chi_pred_test_phys_corrected
)

In [ ]:
# ============================================================
# Regional point-prediction diagnostics
# ============================================================

region_masks = {
    "negative_extreme": chi_true_test_phys < -0.5,
    "negative_moderate": (
        (chi_true_test_phys >= -0.5)
        & (chi_true_test_phys < -0.2)
    ),
    "central": np.abs(chi_true_test_phys) <= 0.2,
    "positive_moderate": (
        (chi_true_test_phys > 0.2)
        & (chi_true_test_phys <= 0.5)
    ),
    "positive_extreme": chi_true_test_phys > 0.5,
}

regional_recalibration_rows = []

for region_name, mask in region_masks.items():
    for model_name, prediction in {
        "original": chi_pred_test_phys_original,
        "linear_recalibrated": chi_pred_test_phys_corrected,
    }.items():

        residual = (
            chi_true_test_phys[mask]
            - prediction[mask]
        )

        abs_error = np.abs(residual)

        regional_recalibration_rows.append({
            "region": region_name,
            "model": model_name,
            "n_samples": int(mask.sum()),
            "true_mean": float(
                np.mean(chi_true_test_phys[mask])
            ),
            "pred_mean": float(
                np.mean(prediction[mask])
            ),
            "bias_true_minus_pred": float(
                np.mean(residual)
            ),
            "MAE": float(np.mean(abs_error)),
            "RMSE": float(
                np.sqrt(np.mean(residual**2))
            ),
            "q90_abs_error": float(
                np.quantile(abs_error, 0.90)
            ),
        })

regional_recalibration_df = pd.DataFrame(
    regional_recalibration_rows
)

regional_recalibration_df

In [ ]:
regional_bias_comparison_df = (
    regional_recalibration_df
    .pivot(
        index="region",
        columns="model",
        values=[
            "bias_true_minus_pred",
            "MAE",
            "RMSE",
            "q90_abs_error",
        ],
    )
)

regional_bias_comparison_df

In [ ]:
plt.figure(figsize=(7, 5.5))

plt.hexbin(
    chi_true_test_phys,
    chi_pred_test_phys_corrected,
    gridsize=80,
    mincnt=1,
    bins="log",
)

plt.colorbar(label="log10(count)")

lims = [
    min(
        chi_true_test_phys.min(),
        chi_pred_test_phys_corrected.min(),
    ),
    max(
        chi_true_test_phys.max(),
        chi_pred_test_phys_corrected.max(),
    ),
]

plt.plot(
    lims,
    lims,
    linestyle="--",
    label="Ideal",
    color="red",
)

plt.xlim(lims)
plt.ylim(lims)
plt.xlabel(r"True $\chi_{\mathrm{eff}}$")
plt.ylabel(r"Corrected predicted $\chi_{\mathrm{eff}}$")
plt.title(r"Linearly recalibrated $\chi_{\mathrm{eff}}$")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
regional_bias_plot_df = (
    regional_recalibration_df
    .pivot(
        index="region",
        columns="model",
        values="bias_true_minus_pred",
    )
)

regional_bias_plot_df.plot(
    kind="bar",
    figsize=(9, 5),
)

plt.axhline(0.0, linestyle="--")
plt.ylabel("Mean residual: true - predicted")
plt.title(
    r"Regional bias before and after linear recalibration"
)
plt.grid(True, axis="y")
plt.xticks(rotation=25)
plt.show()

In [ ]:
# ============================================================
# Out-of-range diagnostic
# ============================================================

out_of_range_original = np.mean(
    (chi_pred_test_phys_original < -1.0)
    | (chi_pred_test_phys_original > 1.0)
)

out_of_range_corrected = np.mean(
    (chi_pred_test_phys_corrected < -1.0)
    | (chi_pred_test_phys_corrected > 1.0)
)

print(
    "Original predictions outside [-1, 1]:",
    out_of_range_original,
)

print(
    "Corrected predictions outside [-1, 1]:",
    out_of_range_corrected,
)

print(
    "Corrected physical prediction range:",
    chi_pred_test_phys_corrected.min(),
    chi_pred_test_phys_corrected.max(),
)

In [ ]:
# ============================================================
# Corrected arrays for conformal calibration
# ============================================================

pred_cal_chi_corrected_2d = (
    pred_cal_chi_corrected[:, None]
)

pred_test_chi_corrected_2d = (
    pred_test_chi_corrected[:, None]
)

y_cal_chi_2d = y_cal_chi_1d[:, None]
y_test_chi_2d = y_test_chi_1d[:, None]

print(pred_cal_chi_corrected_2d.shape)
print(pred_test_chi_corrected_2d.shape)

In [ ]:
# ============================================================
# Global asymmetric conformal after linear recalibration
# ============================================================

corrected_residuals_cal = (
    y_cal_chi_2d
    - pred_cal_chi_corrected_2d
)

global_grouped_residuals = np.empty(
    (1, 1),
    dtype=object,
)

global_grouped_residuals[0, 0] = (
    corrected_residuals_cal[:, 0]
)

global_corrected_calibrator = (
    ConformalIntervalCalibrator(
        confidence_level=confidence_level,
        interval_mode="asymmetric",
        min_samples_per_bin=10,
    )
)

global_corrected_calibrator.fit(
    global_grouped_residuals
)

global_corrected_bin_indices_test = np.zeros(
    y_test_chi_2d.shape,
    dtype=int,
)

(
    global_corrected_lower,
    global_corrected_upper,
) = apply_indices(
    values=pred_test_chi_corrected_2d,
    bin_indices=global_corrected_bin_indices_test,
    intervals=global_corrected_calibrator.intervals_,
)

global_corrected_metrics = CoverageEvaluator(
    confidence_level=confidence_level,
    tolerance_sigmas=(1, 2, 3),
).evaluate_intervals(
    y=y_test_chi_2d,
    lower_bound=global_corrected_lower,
    upper_bound=global_corrected_upper,
    bin_indices=global_corrected_bin_indices_test,
    n_bins=1,
)

print(
    "Global corrected coverage:",
    global_corrected_metrics[
        "global_coverage"
    ][0],
)

print(
    "Global corrected median width physical:",
    global_corrected_metrics[
        "global_median_width"
    ][0] * chi_scale,
)

La corrección lineal no debe reemplazar directamente a la predicción CNN si el objetivo principal es minimizar error global.

Pero sí merece la pena conformalizarla porque:

Ha eliminado casi completamente el sesgo direccional que causaba la infracobertura extrema.

La pregunta ahora es si conformal puede compensar el aumento de dispersión central y producir:

cobertura extrema mejor;
cobertura global de 90%;
anchura aceptable.

### Inspect recalibration on conformal

In [ ]:
def run_global_corrected_conformal(interval_mode):
    corrected_residuals_cal = (
        y_cal_chi_2d
        - pred_cal_chi_corrected_2d
    )

    grouped_residuals = np.empty(
        (1, 1),
        dtype=object,
    )

    grouped_residuals[0, 0] = (
        corrected_residuals_cal[:, 0]
    )

    calibrator = ConformalIntervalCalibrator(
        confidence_level=confidence_level,
        interval_mode=interval_mode,
        min_samples_per_bin=10,
    )

    calibrator.fit(grouped_residuals)

    bin_indices_test = np.zeros(
        y_test_chi_2d.shape,
        dtype=int,
    )

    lower, upper = apply_indices(
        values=pred_test_chi_corrected_2d,
        bin_indices=bin_indices_test,
        intervals=calibrator.intervals_,
    )

    evaluator = CoverageEvaluator(
        confidence_level=confidence_level,
        tolerance_sigmas=(1, 2, 3),
    )

    metrics = evaluator.evaluate_intervals(
        y=y_test_chi_2d,
        lower_bound=lower,
        upper_bound=upper,
        bin_indices=bin_indices_test,
        n_bins=1,
    )

    return {
        "lower": lower,
        "upper": upper,
        "metrics": metrics,
        "calibrator": calibrator,
        "interval_mode": interval_mode,
    }

In [ ]:
corrected_global_results = {}

for interval_mode in [
    "symmetric",
    "asymmetric",
]:
    corrected_global_results[interval_mode] = (
        run_global_corrected_conformal(
            interval_mode=interval_mode
        )
    )

In [ ]:
# Difficulty with 4 bins


corrected_difficulty_results = {}

for interval_mode in [
    "symmetric",
    "asymmetric",
]:
    corrected_difficulty_results[interval_mode] = (
        run_mondrian_regression(
            pred_cal=pred_cal_chi_corrected_2d,
            pred_test=pred_test_chi_corrected_2d,
            y_cal=y_cal_chi_2d,
            y_test=y_test_chi_2d,
            n_bins=4,
            cal_embedding=emb_cal,
            target_embedding=emb_test,
            n_neighbors=5,
            confidence_level=confidence_level,
            apply_jitter=True,
            jitter_variation=1e-10,
            interval_mode=interval_mode,
            taxonomy_mode="difficulty",
            min_samples_per_bin=10,
            tolerance_sigmas=(1, 2, 3),
        )
    )

In [ ]:
def evaluate_corrected_physical_coverage(
    lower_std,
    upper_std,
):
    lower_phys = (
        lower_std[:, 0] * chi_scale
        + chi_mean
    )

    upper_phys = (
        upper_std[:, 0] * chi_scale
        + chi_mean
    )

    covered = (
        (chi_true_test_phys >= lower_phys)
        & (chi_true_test_phys <= upper_phys)
    )

    width = upper_phys - lower_phys

    rows = []

    for bin_idx in range(len(chi_edges) - 1):
        mask = chi_true_bin_idx_hybrid == bin_idx
        n_bin = int(mask.sum())

        coverage = float(
            np.mean(covered[mask])
        )

        sigma = np.sqrt(
            confidence_level
            * (1.0 - confidence_level)
            / n_bin
        )

        lower_miss = float(np.mean(
            chi_true_test_phys[mask]
            < lower_phys[mask]
        ))

        upper_miss = float(np.mean(
            chi_true_test_phys[mask]
            > upper_phys[mask]
        ))

        rows.append({
            "bin_idx": bin_idx,
            "chi_low": chi_edges[bin_idx],
            "chi_high": chi_edges[bin_idx + 1],
            "chi_center": 0.5 * (
                chi_edges[bin_idx]
                + chi_edges[bin_idx + 1]
            ),
            "n_samples": n_bin,
            "coverage": coverage,
            "lower_2sigma": (
                confidence_level - 2.0 * sigma
            ),
            "upper_2sigma": (
                confidence_level + 2.0 * sigma
            ),
            "undercovered_2sigma": (
                coverage
                < confidence_level - 2.0 * sigma
            ),
            "median_width_phys": float(
                np.median(width[mask])
            ),
            "lower_miss_rate": lower_miss,
            "upper_miss_rate": upper_miss,
            "tail_miss_imbalance": abs(
                lower_miss - upper_miss
            ),
        })

    return pd.DataFrame(rows)

In [ ]:
corrected_conformal_rows = []
corrected_physical_tables = {}

for interval_mode, result in (
    corrected_global_results.items()
):
    key = ("global", interval_mode)

    physical_df = (
        evaluate_corrected_physical_coverage(
            result["lower"],
            result["upper"],
        )
    )

    corrected_physical_tables[key] = physical_df

    metrics = result["metrics"]

    corrected_conformal_rows.append({
        "taxonomy": "global",
        "interval_mode": interval_mode,
        "n_bins": 1,
        "global_coverage": float(
            metrics["global_coverage"][0]
        ),
        "median_width_phys": float(
            metrics["global_median_width"][0]
            * chi_scale
        ),
        "negative_extreme_coverage": float(
            physical_df.loc[
                physical_df["bin_idx"] == 0,
                "coverage",
            ].iloc[0]
        ),
        "positive_extreme_coverage": float(
            physical_df.loc[
                physical_df["bin_idx"] == 7,
                "coverage",
            ].iloc[0]
        ),
        "min_physical_coverage": float(
            physical_df["coverage"].min()
        ),
        "n_physical_undercovered_2sigma": int(
            physical_df[
                "undercovered_2sigma"
            ].sum()
        ),
        "max_tail_imbalance": float(
            physical_df[
                "tail_miss_imbalance"
            ].max()
        ),
    })


for interval_mode, result in (
    corrected_difficulty_results.items()
):
    key = ("difficulty", interval_mode)

    physical_df = (
        evaluate_corrected_physical_coverage(
            result.lower,
            result.upper,
        )
    )

    corrected_physical_tables[key] = physical_df

    metrics = result.metrics

    corrected_conformal_rows.append({
        "taxonomy": "difficulty",
        "interval_mode": interval_mode,
        "n_bins": 4,
        "global_coverage": float(
            metrics["global_coverage"][0]
        ),
        "median_width_phys": float(
            metrics["global_median_width"][0]
            * chi_scale
        ),
        "negative_extreme_coverage": float(
            physical_df.loc[
                physical_df["bin_idx"] == 0,
                "coverage",
            ].iloc[0]
        ),
        "positive_extreme_coverage": float(
            physical_df.loc[
                physical_df["bin_idx"] == 7,
                "coverage",
            ].iloc[0]
        ),
        "min_physical_coverage": float(
            physical_df["coverage"].min()
        ),
        "n_physical_undercovered_2sigma": int(
            physical_df[
                "undercovered_2sigma"
            ].sum()
        ),
        "max_tail_imbalance": float(
            physical_df[
                "tail_miss_imbalance"
            ].max()
        ),
    })


corrected_conformal_summary_df = pd.DataFrame(
    corrected_conformal_rows
)

corrected_conformal_summary_df

La anchura mediana aumenta de:

0.6515 → 0.6749

Diferencia:

+0.0234

Incremento relativo aproximado:

0.0234 / 0.6515 ≈ 3.6%

Es un coste bastante moderado comparado con la mejora extrema.

Por tanto, el intercambio es favorable:

+3.6% de anchura
a cambio de
+15–18 puntos de cobertura extrema

La taxonomía por dificultad recupera eficiencia:

0.742 → 0.675

aproximadamente un 9% menos de anchura que el conformal global corregido.

Esto demuestra que la combinación útil es:

corrección lineal del predictor
+
Mondrian por dificultad

No era necesario el híbrido prediction × difficulty.

La corrección lineal elimina la mayor parte de la infracobertura extrema, aunque persisten desviaciones locales moderadas respecto al 90% nominal.

CONCLUSION

El principal origen de la infracobertura física extrema no era una falta de resolución de la taxonomía Mondrian, sino el sesgo de contracción del predictor puntual. Una recalibración lineal ajustada en validación reduce drásticamente ese sesgo y eleva la cobertura extrema desde aproximadamente 72–76% hasta aproximadamente 89–91%. Combinada con una taxonomía de dificultad de 4 bins, mantiene cobertura global cercana al 90% con un incremento moderado de anchura de aproximadamente 3.6%.

### Conclusions from the $\chi_{\mathrm{eff}}$ diagnostic

The 500k CNN exhibits a clear regression-to-the-mean effect for effective
spin. Although the selected difficulty-based Mondrian configuration achieves
approximately 90% global coverage, coverage is strongly heterogeneous across
the true chi_eff range, with substantial undercoverage at both physical
extremes.

Increasing taxonomy resolution and combining predicted $\chi_{eff}$ with embedding-
based difficulty produced well-calibrated operational groups but did not
improve coverage in the extreme true-$\chi_{eff}$ regions. This indicates that the
dominant limitation is the systematic bias of the point predictor rather than
insufficient Mondrian binning resolution.

A linear recalibration fitted exclusively on the validation split substantially
reduced the extreme bias and increased extreme-region coverage to approximately
89–91%, at the cost of worse global point-prediction metrics and moderately
wider intervals. It is therefore retained as a diagnostic sensitivity result,
not as the primary final model.

Future models should first be evaluated for conditional bias and regression
toward the mean before performing extensive Mondrian taxonomy optimization.